# 🌀 12 · Eigenvalues and eigenvectors
### The directions a matrix only stretches — the characteristic equation, 2 × 2 and 3 × 3 examples by hand, five maps of the plane, determinant = product and trace = sum of eigenvalues, triangular and symmetric matrices, the spectral theorem, $B^\top B \succeq 0$, power iteration — and the long-run growth of a population

**BiologyToMath** · Part C — Linear algebra · Module 12 of 28

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/math/tutorial/BiologyToMath/notebooks/12_eigenvalues_eigenvectors.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy (`np.linalg.eig`, `np.linalg.eigh`), SymPy (exact characteristic polynomials and null spaces), Matplotlib |
| **Prerequisites** | Modules 07–11: vectors and dot products, matrix products and determinants by cofactor expansion (Module 08), homogeneous systems, null spaces and the invertible matrix theorem (Module 09), linear independence, basis and rank (Module 11). Module 01 for Python. |
| **Estimated time** | 170 min (reading + running + the ✏️ exercises) |
| **02-680 topics covered** | Eigenvalues and eigenvectors: the definition $A\mathbf x = \lambda\mathbf x$, $\mathbf x\neq\mathbf 0$ (eigenelements) and its geometric meaning; verifying eigenvectors; finding eigenelements — $(A-\lambda I)\mathbf x = \mathbf 0$, why $A-\lambda I$ must be singular, the characteristic polynomial $f(\lambda)=\det(A-\lambda I_n)$ and the characteristic equation, the general procedure; 2 × 2 and 3 × 3 worked examples (cofactor expansion); the 2-D intuition of five linear maps; linear independence of eigenvectors with distinct eigenvalues; determinant = product of eigenvalues, trace = sum, rank and non-zero eigenvalues, eigenvalues of $A^{-1}$; triangular matrices (determinant and eigenvalues from the diagonal; a two-dimensional eigenspace); symmetric matrices and the spectral theorem $A = V\Lambda V^\top$; $B^\top B$ is symmetric positive semidefinite |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. The Leslie population matrices are **illustrative** typed-in numbers, not measurements of a real species. The gene-expression values come from the course's **SIMULATED** infection experiment (`simulate_tissue_expression`), the inflammation network is the toolkit's simplified, illustrative cartoon of innate-immune signalling, and the DNA sequence is **SIMULATED** from a Markov chain whose transition probabilities are rounded from Durbin *et al.* (1998).

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **State** the definition $A\mathbf x = \lambda\mathbf x$ with $\mathbf x \neq \mathbf 0$ and **verify** with one
   matrix–vector product whether a given vector is an eigenvector, and with which eigenvalue.
2. **Derive** the characteristic equation $\det(A-\lambda I) = 0$ from $A\mathbf x = \lambda\mathbf x$ and **explain** why
   $A-\lambda I$ must be *singular* — not zero.
3. **Compute** all eigenvalues and eigenvectors of 2 × 2 and 3 × 3 matrices by hand (determinant, factoring, null space)
   and **check** them with SymPy and NumPy.
4. **Picture** eigenvectors as the directions a linear map only stretches or flips, and **recognise** complex eigenvalues
   (rotations) and repeated eigenvalues with too few eigenvectors (shears).
5. **Prove** that eigenvectors with distinct eigenvalues are linearly independent, and **use** $\det A = \prod_i\lambda_i$,
   $\operatorname{tr}A = \sum_i\lambda_i$ and the eigenvalues of $A^{-1}$, $A^k$ and $A + cI$.
6. **Read off** the eigenvalues of a triangular matrix and **find** a two-dimensional eigenspace.
7. **Apply** the spectral theorem $A = V\Lambda V^\top$ to symmetric matrices, **prove** that $B^\top B$ is positive
   semidefinite, and **interpret** the eigenvectors of a gene–gene covariance matrix as the axes of the data cloud.
8. **Implement** power iteration, **explain** when it converges, and **use** a dominant eigenvector to predict the
   long-run growth and age structure of a population, the hubs of a gene network and the base composition of DNA.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit, project a three-age-class population 14 years ahead | 5 min |
| 1 · What an eigenvector is | definition, collinearity pictures, verifying candidate eigenvectors | 15 min |
| 2 · The characteristic equation | why $A-\lambda I$ must be singular; 2 × 2 examples by hand and with SymPy | 20 min |
| 3 · A 3 × 3 example | cofactor expansion, factoring a cubic, all three eigenvectors; the population matrix | 25 min |
| 4 · Five maps of the plane | stretch, shear, rotation, collapse, symmetric stretch: real, repeated and complex eigenvalues | 15 min |
| 5 · Properties | independence, determinant = product, trace = sum, rank, $A^{-1}$, $A^k$, $A + cI$ | 20 min |
| 6 · Triangular matrices | eigenvalues on the diagonal; a two-dimensional eigenspace | 15 min |
| 7 · Symmetric matrices | spectral theorem, $B^\top B \succeq 0$, the covariance ellipse of two interferon genes | 20 min |
| 8 · Computing eigenvectors | `eig` vs `eigh`, power iteration, hub genes, the stationary base composition of DNA | 20 min |
| 9 · Population growth | Leslie matrices: growth rate, stable age structure, a conservation scenario | 10 min |
| Wrap-up | takeaways, checkpoint, questions, lab note | 5 min |

## 🧫 The biological question

An ecologist follows an insect population that lives at most three years. Every year she counts the animals in three
**age classes** — juveniles (age 0), one-year-olds and two-year-olds — and from marked individuals she knows the
**vital rates**: a juvenile has no offspring, a one-year-old produces 6 surviving juveniles on average and a two-year-old
8; half of the juveniles survive to their second year and half of the one-year-olds survive to their third. The next
year's counts are then a **matrix times this year's counts**,

$$\begin{pmatrix} n_0 \\ n_1 \\ n_2\end{pmatrix}_{t+1} =
\underbrace{\begin{pmatrix} 0 & 6 & 8 \\ \tfrac12 & 0 & 0 \\ 0 & \tfrac12 & 0 \end{pmatrix}}_{L}
\begin{pmatrix} n_0 \\ n_1 \\ n_2\end{pmatrix}_{t}.$$

*In words:* the first row adds up the offspring of all age classes (fecundities 0, 6, 8); the entries below the diagonal
move survivors one age class up (survival ½ and ½). This **Leslie matrix** (Leslie 1945) is the standard model of
structured populations in ecology, conservation biology and fisheries.

The questions a biologist asks — *will the population grow or shrink, how fast, and what fraction of the animals will be
juveniles in the long run?* — have one answer: after a few years the whole population is multiplied by the **same
factor** every year and the **age proportions stop changing**. A vector that a matrix only multiplies by a number is an
**eigenvector**, and the number is its **eigenvalue**. The same idea answers questions all over biology:

| Biological question | Mathematical tool | Where in this module |
|---|---|---|
| Will the population grow, and how fast? | the dominant eigenvalue of the Leslie matrix | §0, §3, §9 |
| Which age structure will it settle into? | the corresponding eigenvector | §1, §9 |
| Along which direction do two co-regulated genes vary most across samples? | eigenvectors of their 2 × 2 covariance matrix (the first step of PCA) | §7 |
| Which genes are the hubs of a regulatory network? | the leading eigenvector of the adjacency matrix (eigenvector centrality) | §8 |
| Which base composition does a DNA Markov chain settle into? | the eigenvector with eigenvalue 1 (stationary distribution) | §8 |
| Does a perturbed equilibrium return or run away? | whether the eigenvalues are smaller or larger than 1 in size | §9, [Module 15 · ⛰️ Gradients, Jacobians, optimization and dynamics](15_multivariable_calculus_dynamics.ipynb) |

> 🧪 **Simulated data, with a known truth.** The population matrices are illustrative numbers typed in by hand. The
> expression values of the interferon genes STAT1 and MX1 come from the course's simulated infection experiment
> (spleen, kidney and liver × control/infected × 4 replicates), whose infection-response genes are built to rise
> together; the regulatory network is the toolkit's cartoon of innate-immune signalling; the DNA sequence is simulated
> from a Markov chain. Because we built the data, we can check that the eigenvectors find what we put in.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sympy", "sympy")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sympy as sp

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py). They draw 2-D arrows (`plot_vectors`), print matrices neatly (`format_matrix`), simulate the infection experiment's expression matrix, build and draw the inflammation network, and simulate DNA from the background and CpG-island Markov chains. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import itertools
from fractions import Fraction

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def circle_layout(nodes: list) -> dict:
    """Place nodes evenly on a circle (first node at the top, clockwise)."""
    n = len(nodes)
    angles = np.pi / 2 - 2 * np.pi * np.arange(n) / max(n, 1)
    return {v: (float(np.cos(a)), float(np.sin(a))) for v, a in zip(nodes, angles)}


def adjacency_matrix(nodes: list, edges: list, directed: bool = False) -> np.ndarray:
    """Adjacency matrix A with A[i, j] = number of edges from nodes[i] to nodes[j].

    For an undirected graph every edge is counted in both directions (A is symmetric);
    a self-loop {v, v} adds 1 to the diagonal entry A[v, v].  List each undirected edge once:
    giving both (u, v) and (v, u) with ``directed=False`` counts that edge twice (a multigraph).
    """
    index = {v: i for i, v in enumerate(nodes)}
    A = np.zeros((len(nodes), len(nodes)), dtype=int)
    for u, v in edges:
        A[index[u], index[v]] += 1
        if not directed and u != v:
            A[index[v], index[u]] += 1
    return A


def draw_graph(edges: list, nodes=None, directed: bool = False, pos=None, ax=None, node_colors=None,
               edge_labels=None, title: str | None = None, font_size: int = 9):
    """Draw a small graph with matplotlib only (no extra packages needed).

    ``edges`` is a list of (u, v) pairs; ``nodes`` (optional) adds isolated nodes and fixes the
    order; ``pos`` maps node → (x, y) (default: ``circle_layout``).  Directed edges get arrows,
    self-loops are drawn as small rings above the node, ``edge_labels`` maps (u, v) → text (e.g. a
    weight) and ``node_colors`` maps node → colour.  Node circles grow with the length of their
    label.  Returns the Axes.
    """
    from matplotlib.transforms import offset_copy

    if nodes is None:
        nodes = []
        for u, v in edges:
            for w in (u, v):
                if w not in nodes:
                    nodes.append(w)
    pos = circle_layout(list(nodes)) if pos is None else pos
    if ax is None:
        _, ax = plt.subplots(figsize=(4.2, 3.6))
    radius = {v: max(0.62 * font_size * len(str(v)) / 2 + 4, 0.9 * font_size) for v in nodes}  # points
    for u, v in edges:
        if u == v:
            ring = offset_copy(ax.transData, fig=ax.figure, x=0, y=1.25 * radius[u], units="points")
            ax.scatter([pos[u][0]], [pos[u][1]], s=(1.3 * radius[u]) ** 2, facecolors="none",
                       edgecolors="0.35", linewidths=1.3, transform=ring, zorder=1)
            continue
        style = "-|>" if directed else "-"
        ax.annotate("", xy=pos[v], xytext=pos[u],
                    arrowprops=dict(arrowstyle=style, color="0.35", lw=1.3, mutation_scale=12,
                                    shrinkA=radius[u], shrinkB=radius[v],
                                    connectionstyle="arc3,rad=0.08" if directed else "arc3,rad=0"))
        if edge_labels and (u, v) in edge_labels:
            (x1, y1), (x2, y2) = pos[u], pos[v]
            ax.text((x1 + x2) / 2, (y1 + y2) / 2, str(edge_labels[(u, v)]), fontsize=font_size - 1,
                    ha="center", va="center", color="darkred", zorder=4,
                    bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.85))
    xs = [pos[v][0] for v in nodes]
    ys = [pos[v][1] for v in nodes]
    colours = [(node_colors or {}).get(v, "#cfe2f3") for v in nodes]
    ax.scatter(xs, ys, s=[(2 * radius[v]) ** 2 for v in nodes], c=colours, edgecolors="0.2",
               linewidths=1.2, zorder=2, clip_on=False)
    for v in nodes:
        ax.text(pos[v][0], pos[v][1], str(v), ha="center", va="center", fontsize=font_size, zorder=3,
                clip_on=False)
    ax.margins(0.22)
    ax.set_aspect("equal", adjustable="datalim")
    ax.axis("off")
    if title:
        ax.set_title(title)
    return ax


def plot_vectors(vectors: list, labels=None, colors=None, ax=None, lim=None, title: str | None = None,
                 origin=(0.0, 0.0)):
    """Draw 2-D vectors as arrows starting at ``origin`` (default: the origin of the plane)."""
    if ax is None:
        _, ax = plt.subplots(figsize=(4.0, 4.0))
    default = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple", "tab:brown"]
    colors = colors or [default[i % len(default)] for i in range(len(vectors))]
    ox, oy = origin
    for i, v in enumerate(vectors):
        ax.quiver(ox, oy, v[0], v[1], angles="xy", scale_units="xy", scale=1, color=colors[i], width=0.012)
        if labels:
            ax.text(ox + v[0] * 1.07, oy + v[1] * 1.07, labels[i], color=colors[i], fontsize=10,
                    ha="center", va="center")
    if lim is None:
        m = max([abs(c) for v in vectors for c in (ox + v[0], oy + v[1])] + [1.0]) * 1.25
        lim = (-m, m)
    ax.set_xlim(*lim)
    ax.set_ylim(*lim)
    ax.axhline(0, color="0.6", lw=0.8)
    ax.axvline(0, color="0.6", lw=0.8)
    ax.set_aspect("equal")
    ax.grid(alpha=0.3)
    if title:
        ax.set_title(title)
    return ax


def format_matrix(M, n_coef=None, width: int = 7) -> str:
    """Pretty-print a matrix of numbers, Fractions or SymPy rationals (complex entries with a zero
    imaginary part print as reals); a bar ``|`` is drawn after ``n_coef`` columns."""
    M = np.asarray(M, dtype=object)

    def fmt(x):
        if getattr(x, "is_Rational", False):            # a SymPy Rational/Integer → exact fraction
            x = Fraction(int(x.p), int(x.q))
        if isinstance(x, Fraction):
            return str(x.numerator) if x.denominator == 1 else f"{x.numerator}/{x.denominator}"
        if isinstance(x, (int, np.integer)):
            return str(int(x))
        if isinstance(x, (complex, np.complexfloating)):
            if abs(x.imag) > 1e-12:
                return f"{x.real:.3g}{x.imag:+.3g}i"
            x = x.real
        x = float(x)
        return "0" if abs(x) < 1e-12 else f"{x:.4g}"

    lines = []
    for row in M:
        cells = [fmt(x).rjust(width) for x in row]
        if n_coef is not None and n_coef < len(cells):
            cells = cells[:n_coef] + ["  |"] + cells[n_coef:]
        lines.append("[" + "".join(cells) + " ]")
    return "\n".join(lines)


NUCLEOTIDES = ("A", "C", "G", "T")


# First-order Markov-chain transition probabilities P(next | current) for DNA, rows = current base,
# columns = next base, both in the order A, C, G, T.  Rounded to two decimals from the classic
# CpG-island ("+") and background ("−") models of Durbin et al. (1998), Table 3.1, which were
# estimated from human sequence.  In CpG islands a C is often followed by a G; elsewhere CG is rare
# because methylated C in CpG mutates to T.
CPG_ISLAND_TRANSITIONS = np.array([
    [0.18, 0.27, 0.43, 0.12],
    [0.17, 0.37, 0.27, 0.19],
    [0.16, 0.34, 0.37, 0.13],
    [0.08, 0.36, 0.38, 0.18],
])


BACKGROUND_TRANSITIONS = np.array([
    [0.30, 0.20, 0.29, 0.21],
    [0.32, 0.30, 0.08, 0.30],
    [0.25, 0.25, 0.29, 0.21],
    [0.18, 0.24, 0.29, 0.29],
])


def simulate_dna(length: int, transitions=None, initial=None, seed: int = 0) -> str:
    """Random DNA string.  Without ``transitions`` the bases are i.i.d. uniform; otherwise they
    follow a first-order Markov chain with the given 4 × 4 transition matrix (order A, C, G, T)
    and initial distribution ``initial`` (default uniform)."""
    rng = np.random.default_rng(seed)
    initial = np.full(4, 0.25) if initial is None else np.asarray(initial, dtype=float)
    if transitions is None:
        return "".join(rng.choice(list(NUCLEOTIDES), size=length, p=initial))
    P = np.asarray(transitions, dtype=float)
    states = [rng.choice(4, p=initial)]
    for _ in range(length - 1):
        states.append(rng.choice(4, p=P[states[-1]]))
    return "".join(NUCLEOTIDES[s] for s in states)


# Named genes of the simulated infection experiment.  The infection-response genes are real genes
# known to be induced during infection (cytokines, interferon-stimulated genes, acute-phase
# proteins; lysozyme, LYZ, is an antimicrobial enzyme rather than a classic acute-phase protein and is
# grouped with them only for simplicity); the reference genes are classic "housekeeping" genes used
# to normalise qPCR.
INFECTION_PATHWAYS = {
    "cytokine signalling": ["IL1B", "TNF", "IL6", "CXCL8"],
    "interferon response": ["IRF1", "STAT1", "MX1", "ISG15"],
    "acute phase": ["SAA1", "HAMP", "C3", "LYZ"],
}


HOUSEKEEPING_GENES = ["ACTB", "GAPDH", "EEF1A1", "RPL13A", "TBP", "HPRT1",
                      "PPIA", "UBC", "YWHAZ", "SDHA", "TUBA1A", "B2M"]


# A simplified, illustrative cartoon of innate-immune signalling among the named genes
# (regulator → target).  NF-κB (NFKB1) induces cytokines; TNF and IL-1β activate NF-κB (a feedback
# loop); IL-6 acts through STAT3 to induce acute-phase genes; STAT1 induces interferon-stimulated genes.
INFLAMMATION_NETWORK_EDGES = [
    ("NFKB1", "IL1B"), ("NFKB1", "TNF"), ("NFKB1", "IL6"), ("NFKB1", "CXCL8"),
    ("TNF", "NFKB1"), ("IL1B", "NFKB1"), ("IL6", "STAT3"), ("STAT3", "SAA1"),
    ("STAT3", "HAMP"), ("STAT3", "C3"), ("STAT1", "IRF1"), ("STAT1", "MX1"),
    ("STAT1", "ISG15"), ("IRF1", "ISG15"), ("NFKB1", "LYZ"),
]


def simulate_tissue_expression(n_genes: int = 400, tissues=("spleen", "kidney", "liver"), n_reps: int = 4,
                               seed: int = 2680) -> dict:
    """SIMULATED log2 gene expression from an infection experiment in several tissues.

    Every tissue has ``n_reps`` control and ``n_reps`` infected samples.  The 12 genes of
    ``INFECTION_PATHWAYS`` respond to infection in every tissue (up by 1.5–3 log2 units); further
    "g…" genes respond in one tissue only or in two tissues, so the sets of truly changed genes
    overlap like the differentially-expressed-gene sets of a real multi-tissue study.  The 12
    ``HOUSEKEEPING_GENES`` are stable (tiny noise, no response).  About a third of all genes differ in
    baseline level between tissues.

    Returns a dict with ``X`` (samples × genes, log2 expression), ``samples`` (DataFrame with
    columns sample, tissue, condition), ``genes`` (list), ``true_lfc`` (DataFrame tissues × genes,
    true infected − control difference), ``true_de`` (dict tissue → set of truly changed genes),
    ``pathways`` (dict pathway → genes) and ``source``.
    """
    rng = np.random.default_rng(seed)
    named = [g for genes in INFECTION_PATHWAYS.values() for g in genes]
    n_other = n_genes - len(named) - len(HOUSEKEEPING_GENES)
    if n_other < 60:
        raise ValueError("n_genes must be at least 84")
    genes = named + HOUSEKEEPING_GENES + [f"g{i:03d}" for i in range(1, n_other + 1)]
    J, T = len(genes), len(tissues)
    base = rng.normal(7.0, 1.8, J)
    base[len(named):len(named) + len(HOUSEKEEPING_GENES)] = rng.uniform(9.0, 12.0, len(HOUSEKEEPING_GENES))
    tissue_shift = rng.normal(0.0, 1.0, (T, J)) * (rng.random(J) < 0.35)
    tissue_shift[:, len(named):len(named) + len(HOUSEKEEPING_GENES)] = 0.0
    lfc = np.zeros((T, J))
    lfc[:, :len(named)] = rng.uniform(1.5, 3.0, (T, len(named)))
    other = list(range(len(named) + len(HOUSEKEEPING_GENES), J))
    rng.shuffle(other)
    sizes_single = [24, 18, 14, 12, 10][:T]
    pairs = list(itertools.combinations(range(T), 2))
    sizes_pair = [10, 5, 4, 3, 3, 2, 2, 2, 2, 2][:len(pairs)]
    k = 0

    def effect(m):
        return rng.uniform(1.0, 2.5, m) * np.where(rng.random(m) < 0.7, 1.0, -1.0)

    for t, m in enumerate(sizes_single):
        idx = other[k:k + m]
        lfc[t, idx] = effect(m)
        k += m
    for (t1, t2), m in zip(pairs, sizes_pair):
        idx = other[k:k + m]
        e = effect(m)
        lfc[t1, idx] = e
        lfc[t2, idx] = e * rng.uniform(0.8, 1.2, m)
        k += m
    noise_sd = rng.uniform(0.25, 0.55, J)
    noise_sd[len(named):len(named) + len(HOUSEKEEPING_GENES)] = 0.08
    rows, X = [], []
    for t, tissue in enumerate(tissues):
        for condition in ("control", "infected"):
            for r in range(1, n_reps + 1):
                mean = base + tissue_shift[t] + (lfc[t] if condition == "infected" else 0.0)
                X.append(mean + rng.normal(0.0, noise_sd))
                rows.append((f"{tissue}_{condition[:3]}{r}", tissue, condition))
    samples = pd.DataFrame(rows, columns=["sample", "tissue", "condition"])
    true_lfc = pd.DataFrame(lfc, index=list(tissues), columns=genes)
    true_de = {tissue: {g for g, v in zip(genes, lfc[t]) if v != 0} for t, tissue in enumerate(tissues)}
    return {"X": np.array(X), "samples": samples, "genes": genes, "true_lfc": true_lfc,
            "true_de": true_de, "pathways": {k_: list(v) for k_, v in INFECTION_PATHWAYS.items()},
            "source": "SIMULATED log2 expression (BiologyToMath simulate_tissue_expression)"}

print("🧰 toolkit loaded:", ", ".join(sorted(['BACKGROUND_TRANSITIONS', 'CPG_ISLAND_TRANSITIONS', 'HOUSEKEEPING_GENES', 'INFECTION_PATHWAYS', 'INFLAMMATION_NETWORK_EDGES', 'NUCLEOTIDES', 'adjacency_matrix', 'circle_layout', 'draw_graph', 'format_matrix', 'plot_vectors', 'simulate_dna', 'simulate_tissue_expression'])))

In [ ]:
# ── 0.1 Project the (illustrative) insect population 14 years ahead ──
L = np.array([[0.0, 6.0, 8.0],     # fecundities: offspring per juvenile, 1-year-old, 2-year-old
              [0.5, 0.0, 0.0],     # half of the juveniles survive to age 1
              [0.0, 0.5, 0.0]])    # half of the 1-year-olds survive to age 2
n = np.array([0.0, 0.0, 10.0])     # year 0: ten 2-year-old colonists arrive

rows = [dict(year=0, juveniles=n[0], age1=n[1], age2=n[2], total=n.sum(), growth=np.nan)]
for year in range(1, 15):
    n_new = L @ n                                        # next year's counts = L × this year's counts
    rows.append(dict(year=year, juveniles=n_new[0], age1=n_new[1], age2=n_new[2],
                     total=n_new.sum(), growth=n_new.sum() / n.sum()))
    n = n_new
pop = pd.DataFrame(rows).set_index("year")
for col in ["juveniles", "age1", "age2"]:
    pop["% " + col] = 100 * pop[col] / pop["total"]
print(pop.round(3).to_string())

🔍 **What just happened?** Ten two-year-olds arrive in year 0. In year 1 they have produced $8\times10 = 80$ juveniles
and died; in year 2 half of those are one-year-olds (40); in year 3 the 40 one-year-olds produce 240 juveniles and 20 of
them (the survivors) are now two years old — the counts jump around, and the yearly growth factor (column `growth`) swings wildly between 8, 0.5,
6.5 and 1.08. Then something remarkable happens: from about year 9 on the growth factor settles at **2** (2.093, 1.951,
2.028, 1.985, 2.008, 1.996) and the **age proportions freeze** at about 76 % juveniles, 19 % one-year-olds and 4.8 %
two-year-olds. In year 14 the counts are (290,880, 73,000, 18,120), and multiplying this vector by $L$ very nearly
doubles it. Section 1 shows that $(16, 4, 1)$ — whose proportions are
$16/21 = 76.2\,\%$, $4/21 = 19.0\,\%$ and $1/21 = 4.8\,\%$ — is an **eigenvector** of $L$ with **eigenvalue 2**, and
Section 9 explains why (almost) every starting population ends up there.

---
## 1 · What an eigenvector is

> 📍 **Where we are.** In Module 08 a matrix became a machine that **transforms** vectors: $A\mathbf x$ is a new vector,
> usually pointing in a new direction (Module 08 drew how a matrix shears and stretches the unit square). In
> Modules 09–11 you solved homogeneous systems $M\mathbf x = \mathbf 0$ and learned that they have non-zero solutions
> exactly when $M$ is singular. This module asks a new question: **are there special directions that the matrix does
> not turn, but only stretches, shrinks or flips?** These directions — eigenvectors — are the skeleton of a matrix:
> they explain its long-run behaviour (populations, Markov chains), its geometry (PCA) and its stability (Module 15).

**Definition.** Let $A\in\mathbb R^{n\times n}$ be a **square** matrix. A number $\lambda\in\mathbb R$ is an
**eigenvalue** of $A$ and a vector $\mathbf x\in\mathbb R^n\setminus\{\mathbf 0\}$ is an **eigenvector** of $A$ if

$$A\mathbf x = \lambda\mathbf x .$$

*In words:* multiplying the eigenvector by the matrix gives the same vector multiplied by a plain number $\lambda$
(read "lambda"). We say that $\lambda$ is the eigenvalue **for** $\mathbf x$ and $\mathbf x$ is an eigenvector **for**
$\lambda$; together $(\lambda, \mathbf x)$ are called **eigenelements** (or an **eigenpair**). *Eigen* is German for
"own" or "characteristic": the eigenvectors are the matrix's own directions.

Four remarks you will use constantly:

* **$\mathbf x = \mathbf 0$ is excluded** ($\mathbb R^n\setminus\{\mathbf 0\}$ means "all vectors of $\mathbb R^n$ except
  the zero vector"). Without that rule every number would be an eigenvalue, because $A\mathbf 0 = \lambda\mathbf 0$ for
  every $\lambda$.
* **$\lambda = 0$ is allowed.** Then $A\mathbf x = \mathbf 0$ with $\mathbf x\neq\mathbf 0$: the eigenvectors for 0 are the
  non-zero vectors of the null space, and $A$ is singular (Module 09).
* **Only the direction matters.** If $\mathbf x$ is an eigenvector, so is $c\,\mathbf x$ for every $c\neq0$:
  $A(c\mathbf x) = c\,A\mathbf x = c\lambda\mathbf x = \lambda(c\mathbf x)$. An eigenvector is really an **eigen-line**
  through the origin; we pick a convenient representative (integers, or length 1).
* **Geometric meaning.** $A\mathbf x = \lambda\mathbf x$ says that $A\mathbf x$ and $\mathbf x$ lie **on the same line
  through the origin** (they are *collinear* with $\mathbf 0$). $\lambda > 1$ stretches, $0<\lambda<1$ shrinks, $\lambda<0$
  flips to the other side of the origin.

**A first check.** Take $A = \begin{pmatrix}3&4\\2&1\end{pmatrix}$, $\mathbf x = \begin{pmatrix}2\\1\end{pmatrix}$ and
$\lambda = 5$:

$$A\mathbf x = \begin{pmatrix}3\cdot2 + 4\cdot1\\ 2\cdot2 + 1\cdot1\end{pmatrix} = \begin{pmatrix}10\\5\end{pmatrix}
= 5\begin{pmatrix}2\\1\end{pmatrix} = \lambda\mathbf x .$$

*In words:* the matrix sends $(2,1)$ to $(10,5)$, which points in exactly the same direction, five times as long — so
$(2,1)$ is an eigenvector of $A$ with eigenvalue 5.

In [ ]:
# ── 1.1 Verifying candidate eigenvectors: one product, one comparison ──
def eigen_check(A, x, name=""):
    """Compute A x and decide whether it is a multiple λ x of x (i.e. whether x is an eigenvector).

    λ is estimated as (x · Ax)/(x · x) — the only number that could work — and the relative residual
    ‖Ax − λx‖ / ‖Ax‖ says how far A x is from that multiple (0 for an exact eigenvector)."""
    A, x = np.asarray(A, dtype=float), np.asarray(x, dtype=float)
    Ax = A @ x
    lam = (x @ Ax) / (x @ x)
    residual = np.linalg.norm(Ax - lam * x) / np.linalg.norm(Ax)
    if residual < 1e-12:
        verdict = f"= {lam:g} · x  → eigenvector, λ = {lam:g}"
    elif residual < 0.01:
        verdict = f"≈ {lam:.4f} · x  → almost an eigenvector (relative residual {residual:.1e})"
    else:
        verdict = f"is not a multiple of x → NOT an eigenvector (relative residual {residual:.2f})"
    print(f"{name:26s} x = {x.round(3).tolist()},  Ax = {Ax.round(3).tolist()}  {verdict}")
    return lam if residual < 1e-12 else None


A_notes = np.array([[3, 4], [2, 1]])
A_ex1 = np.array([[2, 2], [-4, 8]])
L_frac = np.array([[0, 6, 8], [0.5, 0, 0], [0, 0.5, 0]])       # the population matrix of Section 0

eigen_check(A_notes, [2, 1], "A = [[3,4],[2,1]]")
eigen_check(A_ex1, [1, 1], "A = [[2,2],[-4,8]], v")
eigen_check(A_ex1, [2, 1], "A = [[2,2],[-4,8]], w")
eigen_check(L_frac, [16, 4, 1], "Leslie L, v")
eigen_check(L_frac, [2, 2, 2], "Leslie L, w")
eigen_check(L_frac, [160, 40, 10], "Leslie L, 10·v")
eigen_check(L_frac, pop.loc[14, ["juveniles", "age1", "age2"]].to_numpy(), "Leslie L, year-14 counts");

🔍 **What just happened?** Seven verdicts, each from a single matrix–vector product (the first line repeats the check
$(2,1)\mapsto(10,5)$ above):

* $A = \begin{pmatrix}2&2\\-4&8\end{pmatrix}$ sends $\mathbf v = (1,1)$ to $(4,4) = 4\mathbf v$ — an eigenvector with
  $\lambda = 4$ — but sends $\mathbf w = (2,1)$ to $(6,0)$, which is not a multiple of $(2,1)$ (a multiple would need
  $6 = 2c$ *and* $0 = 1\cdot c$). So $\mathbf w$ is **not** an eigenvector.
* The population matrix sends $\mathbf v = (16,4,1)$ to $(32, 8, 2) = 2\mathbf v$: an eigenvector with $\lambda = 2$.
  By hand: $0\cdot16 + 6\cdot4 + 8\cdot1 = 32$, $\tfrac12\cdot16 = 8$, $\tfrac12\cdot4 = 2$. But $\mathbf w = (2,2,2)$
  goes to $(28, 1, 1)$ — not a multiple.
* $10\mathbf v = (160, 40, 10)$ is an eigenvector too (only the direction matters). The year-14 counts of Section 0
  are *almost* one: $A\mathbf x$ is $2.0035$ times $\mathbf x$ up to a relative residual of $1.4\times10^{-3}$ —
  the projection of the population has nearly converged onto the eigen-line of $\mathbf v$, and the residual keeps
  shrinking every year (Section 9).

The printed $\lambda$ comes from $\lambda = \dfrac{\mathbf x\cdot A\mathbf x}{\mathbf x\cdot\mathbf x}$ (if $A\mathbf x =
\lambda\mathbf x$, dot both sides with $\mathbf x$ and divide by $\mathbf x\cdot\mathbf x$); the relative residual
$\|A\mathbf x - \lambda\mathbf x\|/\|A\mathbf x\|$ then measures how far $A\mathbf x$ is from that multiple. This ratio is
called the **Rayleigh quotient**; it returns in Section 8.

In [ ]:
# ── 1.2 The picture: Av lies on the line of v, Aw leaves the line of w ──
v, w = np.array([1.0, 1.0]), np.array([2.0, 1.0])
Av, Aw = A_ex1 @ v, A_ex1 @ w

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
ax = axes[0]
t = np.linspace(-1.5, 8, 2)
ax.plot(t * v[0], t * v[1], color="tab:red", lw=1, alpha=0.6, label="line through 0 and v")
ax.plot(t * w[0] / 2, t * w[1] / 2, color="tab:blue", lw=1, alpha=0.6, label="line through 0 and w")
for p, lab, c in [(v, "v", "tab:red"), (Av, "Av = 4v", "tab:red"), (w, "w", "tab:blue"), (Aw, "Aw", "tab:blue")]:
    ax.plot(*p, "o", color=c, ms=7)
    ax.annotate(lab, p, textcoords="offset points", xytext=(6, 6), color=c, fontsize=10)
ax.plot(0, 0, "ko"); ax.annotate("0", (0, 0), textcoords="offset points", xytext=(-12, 4))
ax.set_xlim(-1, 7); ax.set_ylim(-1, 5); ax.set_aspect("equal"); ax.grid(alpha=0.3)
ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$"); ax.legend(loc="upper left", fontsize=8)
ax.set_title("A = [[2, 2], [−4, 8]]: v is an eigenvector, w is not")

# eigen-direction hunt: sweep unit vectors and measure how far A turns them
theta = np.linspace(0, 180, 721)
units = np.stack([np.cos(np.radians(theta)), np.sin(np.radians(theta))])
images = A_notes @ units
sin_turn = (units[0] * images[1] - units[1] * images[0]) / np.linalg.norm(images, axis=0)   # sin(angle x → Ax)
ax = axes[1]
ax.plot(theta, sin_turn, color="tab:purple")
ax.axhline(0, color="0.5", lw=0.8)
for ang, lab in [(np.degrees(np.arctan2(1, 2)), "(2, 1): Ax = 5x"), (135, "(−1, 1): Ax = −x")]:
    ax.plot(ang, 0, "o", color="tab:green")
    ax.annotate(lab, (ang, 0), textcoords="offset points", xytext=(5, 10), color="tab:green", fontsize=9)
ax.set_xlabel("direction θ of the unit vector x (degrees)")
ax.set_ylabel("sin(angle between x and Ax)")
ax.set_title("A = [[3, 4], [2, 1]]: where does A not turn x?")
plt.tight_layout(); plt.show()

🔍 **What just happened?** *Left:* the red line through $\mathbf 0$ and $\mathbf v = (1,1)$ also contains
$A\mathbf v = (4,4)$ — the matrix moved $\mathbf v$ **along its own line**, four times further out. The blue point
$A\mathbf w = (6,0)$ is far off the blue line through $\mathbf w = (2,1)$: the matrix turned $\mathbf w$. This is the
whole geometric content of $A\mathbf x = \lambda\mathbf x$.

*Right:* we swept unit vectors $\mathbf x = (\cos\theta, \sin\theta)$ around half a circle and plotted the sine of the
angle by which $A = \begin{pmatrix}3&4\\2&1\end{pmatrix}$ turns each one. The sine is 0 exactly when $A\mathbf x$ lies
on the line of $\mathbf x$ — pointing the same way (angle 0°) *or* the opposite way (180°). Almost every direction is
turned; the curve crosses 0 at exactly two directions: $\theta \approx 26.6°$, the direction of $(2,1)$, where
$A\mathbf x = 5\mathbf x$, and $\theta = 135°$, the direction of $(-1,1)$, where $A\mathbf x = -\mathbf x$ (flipped
through the origin but still on the same line). Hunting for zeros of such a curve works in 2-D only; Section 2 gives
an algebraic method that works in any dimension.

✏️ **Your turn.** (a) Is $(1, 2)$ an eigenvector of $\begin{pmatrix}2&2\\-4&8\end{pmatrix}$? Use `eigen_check`. (b) Show
by hand that $(-3,-3)$ is an eigenvector of the same matrix, and say why that needs no new calculation. (c) Is
$(4, -2, 1)$ an eigenvector of the population matrix $L$? With which eigenvalue — and what could a *negative* entry
mean for a vector of animal counts?

### 📘 Lecture notes — eigenvalues, eigenvectors and how to verify them

**Definition.** For a square matrix $A\in\mathbb R^{n\times n}$, a scalar $\lambda\in\mathbb R$ and a vector
$\mathbf x\in\mathbb R^n\setminus\{\mathbf 0\}$ with

$$A\mathbf x = \lambda\mathbf x$$

are an **eigenvalue** and an **eigenvector** of $A$; together they are **eigenelements**. If $A\mathbf v = \lambda\mathbf v$
for $\mathbf v\neq\mathbf 0$, $\lambda$ is "the eigenvalue for $\mathbf v$" and $\mathbf v$ "an eigenvector for $\lambda$".

*In words:* the transformation of $\mathbf x$ by $A$ is a scaled version of $\mathbf x$ that lies on the same line
through the origin; the eigenvalue is the scaling factor.

**Interpretation.** $A\mathbf v$ and $\mathbf v$ are **collinear** with the origin. Every non-zero multiple of an
eigenvector is again an eigenvector for the same $\lambda$; $\lambda$ may be negative (flip) or zero ($A$ singular);
$\mathbf x = \mathbf 0$ never counts.

**Worked examples (verify = multiply and compare).**
* $A = \begin{pmatrix}3&4\\2&1\end{pmatrix}$, $\mathbf x = (2,1)$: $A\mathbf x = (3\cdot2+4\cdot1,\ 2\cdot2+1\cdot1) =
  (10,5) = 5\mathbf x$, so $\lambda = 5$.
* $A = \begin{pmatrix}2&2\\-4&8\end{pmatrix}$: $\mathbf v = (1,1) \mapsto (4,4) = 4\mathbf v$ (eigenvector, $\lambda=4$);
  $\mathbf w = (2,1)\mapsto(6,0)$, not a scalar multiple of $\mathbf w$ (not an eigenvector).
* $A = \begin{pmatrix}0&6&8\\\frac12&0&0\\0&\frac12&0\end{pmatrix}$: $\mathbf v = (16,4,1)\mapsto(32,8,2) = 2\mathbf v$
  (eigenvector, $\lambda = 2$); $\mathbf w = (2,2,2)\mapsto(28,1,1)$ (not an eigenvector).

**Common confusions.** (1) "$A\mathbf x$ has the same *length* as $\mathbf x$" — no, the same *direction line*.
(2) A vector is never "the" eigenvector: eigenvectors come as whole lines (or planes, Section 6). (3) Eigenvalues
are defined for **square** matrices only — a $3\times2$ matrix sends $\mathbb R^2$ to $\mathbb R^3$, so $A\mathbf x$
and $\mathbf x$ cannot even be compared (Module 13's singular values fill that gap).

> 🎓 **Exam tip.** To check a candidate eigenvector, compute $A\mathbf x$ and look for one number $\lambda$ that works
> for **every** component: $(6,0)$ vs $(2,1)$ fails because $6/2 \neq 0/1$. Never divide by a zero component of
> $\mathbf x$ without checking that the matching component of $A\mathbf x$ is zero too.

---
## 2 · Finding eigenvalues: the characteristic equation

> 📍 **Where we are.** Verifying a candidate is easy; *finding* eigenvalues and eigenvectors when only $A$ is given
> needs a trick. The trick turns the question into one you already know from Module 09 — when does a homogeneous system
> have a non-zero solution? — and that question into a determinant (Module 08). The price: we must find the roots of a
> **polynomial**, a *nonlinear* equation in the single unknown $\lambda$, so this is no longer "just" a linear system.

Start from the definition and move everything to one side ($I_n$ is the $n\times n$ identity matrix, so
$I_n\mathbf x = \mathbf x$):

$$\begin{aligned}
A\mathbf x &= \lambda\mathbf x\\
A\mathbf x &= \lambda I_n\mathbf x\\
A\mathbf x - \lambda I_n\mathbf x &= \lambda I_n\mathbf x - \lambda I_n\mathbf x\\
(A - \lambda I_n)\,\mathbf x &= \mathbf 0 .
\end{aligned}$$

*In words:* line 2 writes $\lambda\mathbf x$ as the matrix $\lambda I_n$ times $\mathbf x$ (you cannot subtract a number
from a matrix, but you can subtract the matrix $\lambda I_n$, which has $\lambda$ on the diagonal); line 3 subtracts it
from both sides; line 4 factors $\mathbf x$ out on the right. For a fixed number $\lambda$ this is a **homogeneous
linear system** in the $n$ unknowns $x_1,\dots,x_n$ with coefficient matrix $A - \lambda I_n$.

⚠️ **Common confusion — "$\mathbf x\neq\mathbf 0$, so $A - \lambda I_n$ must be 0".** That conclusion is false: a
non-zero matrix can send a non-zero vector to $\mathbf 0$, e.g. $\begin{pmatrix}1&1\\1&1\end{pmatrix}\begin{pmatrix}1\\-1
\end{pmatrix} = \begin{pmatrix}0\\0\end{pmatrix}$. What the equation really requires is that $A - \lambda I_n$ is
**singular** (not invertible). If it were invertible, we could multiply by its inverse and get
$\mathbf x = (A-\lambda I_n)^{-1}\mathbf 0 = \mathbf 0$, which is forbidden. And by the invertible matrix theorem
(Module 09) a square matrix is singular exactly when its **determinant is 0**:

$$\lambda \text{ is an eigenvalue of } A \iff (A-\lambda I_n)\mathbf x = \mathbf 0 \text{ has a solution } \mathbf x\neq\mathbf 0
\iff \det(A - \lambda I_n) = 0 .$$

*In words:* the eigenvalues are precisely the numbers $\lambda$ that make $A - \lambda I_n$ singular.

**Definitions.** The **characteristic polynomial** of $A$ is the function $f(\lambda) = \det(A - \lambda I_n)$ — the
determinant of a matrix whose diagonal entries contain the unknown $\lambda$, which works out to a polynomial of degree
$n$. Setting it to zero gives the **characteristic equation** $\det(A - \lambda I_n) = 0$. Theorem: $\lambda_0$ is an
eigenvalue of $A$ **if and only if** $f(\lambda_0) = 0$. (Many texts write the determinant with bars, $|M| = \det M$, so
the equation reads $|A - \lambda I_n| = 0$.)

🧮 **Math refresher — the 2 × 2 determinant and the quadratic formula.** For
$M = \begin{pmatrix}a&b\\c&d\end{pmatrix}$, $\det M = ad - bc$ (Module 08). A quadratic equation $\lambda^2 + p\lambda +
q = 0$ has the roots $\lambda = \dfrac{-p\pm\sqrt{p^2 - 4q}}{2}$: two real roots if $p^2 - 4q > 0$, one repeated root
if $p^2-4q = 0$, and two complex roots if $p^2 - 4q < 0$ (Section 4). When the roots are integers you can often see
the factorisation directly: $\lambda^2 - 4\lambda - 5 = (\lambda+1)(\lambda-5)$ because $(-5)\cdot1 = -5$ and
$-5 + 1 = -4$.

**The 2 × 2 case in general.** For $A = \begin{pmatrix}a&b\\c&d\end{pmatrix}$,

$$f(\lambda) = \det\begin{pmatrix}a-\lambda & b\\ c & d-\lambda\end{pmatrix} = (a-\lambda)(d-\lambda) - bc
= \lambda^2 - (a+d)\lambda + (ad - bc) = \lambda^2 - \operatorname{tr}(A)\,\lambda + \det A .$$

*In words:* expand the product $(a-\lambda)(d-\lambda) = ad - a\lambda - d\lambda + \lambda^2$ and collect powers of
$\lambda$; the middle coefficient is minus the **trace** (sum of the diagonal, $a+d$) and the constant is the
determinant. Section 5 shows that this is no coincidence.

### 2.1 The worked example, step by step

Find the eigenvalues and eigenvectors of $A = \begin{pmatrix}3&4\\2&1\end{pmatrix}$.

**Step 1 — the characteristic equation.**

$$\det\begin{pmatrix}3-\lambda & 4\\ 2 & 1-\lambda\end{pmatrix} = (3-\lambda)(1-\lambda) - (4)(2)
= (3 - 4\lambda + \lambda^2) - 8 = \lambda^2 - 4\lambda - 5 = (\lambda+1)(\lambda-5) = 0 .$$

*In words:* $A - \lambda I$ is singular exactly when $\lambda = 5$ or $\lambda = -1$; these are the eigenvalues.

**Step 2 — plug each eigenvalue back in and solve $(A - \lambda I)\mathbf x = \mathbf 0$.**

$$\lambda = 5:\quad \begin{pmatrix}-2&4\\2&-4\end{pmatrix}\begin{pmatrix}x_1\\x_2\end{pmatrix} = \mathbf 0
\ \Rightarrow\ -2x_1 + 4x_2 = 0 \ \Rightarrow\ x_1 = 2x_2 \ \Rightarrow\ \mathbf x = \begin{pmatrix}2\\1\end{pmatrix};$$

$$\lambda = -1:\quad \begin{pmatrix}4&4\\2&2\end{pmatrix}\begin{pmatrix}x_1\\x_2\end{pmatrix} = \mathbf 0
\ \Rightarrow\ 4x_1 + 4x_2 = 0 \ \Rightarrow\ x_1 = -x_2 \ \Rightarrow\ \mathbf x = \begin{pmatrix}1\\-1\end{pmatrix}.$$

*In words:* because $A-\lambda I$ is singular, its second row is a multiple of the first, so there is really only one
equation in two unknowns; one unknown is free ($x_2 = t$), and every choice $t\neq0$ gives an eigenvector — we pick
$t = 1$. The trivial solution $\mathbf x = \mathbf 0$ always solves the system but is excluded by definition.

In [ ]:
# ── 2.1 The same example with SymPy: exact characteristic polynomial, roots and null spaces ──
lam = sp.symbols("lambda")
A_sym = sp.Matrix([[3, 4], [2, 1]])

char_matrix = A_sym - lam * sp.eye(2)
f = char_matrix.det()
print("A − λI =", char_matrix.tolist())
print("f(λ) = det(A − λI) =", sp.expand(f), "=", sp.factor(f))
roots = sp.solve(sp.Eq(f, 0), lam)
print("eigenvalues (roots of f):", roots)

for r in roots:
    M = A_sym - r * sp.eye(2)
    basis = M.nullspace()                      # all solutions of (A − rI) x = 0
    vec = basis[0] / sp.gcd(list(basis[0]))   # clear fractions → simplest integer representative
    print(f"\nλ = {r}:  A − λI = {M.tolist()},  rank {M.rank()} (singular!)")
    print(f"   null space spanned by {list(vec)};  check A x = {list(A_sym * vec)} = {r}·{list(vec)}")

🔍 **What just happened?** SymPy computed $f(\lambda) = \lambda^2 - 4\lambda - 5 = (\lambda - 5)(\lambda + 1)$ — the
same polynomial as by hand — and its roots $-1$ and $5$. For each root, $A - \lambda I$ has **rank 1** instead of 2:
it is singular, so its null space is a whole line, spanned by $(-1, 1)$ for $\lambda=-1$ (the same line as our
$(1,-1)$; SymPy simply chose the other sign) and $(2,1)$ for $\lambda = 5$. The last check multiplies back:
$A(2,1)^\top = (10,5)^\top$. SymPy's `nullspace()` is exactly the "solve the homogeneous system" step of Module 09.

### 2.2 The general procedure — and a picture of the characteristic polynomial

For any $A\in\mathbb R^{n\times n}$:

1. Set up $(A - \lambda I_n)\mathbf v = \mathbf 0$ — a linear system in the $n$ unknowns $\mathbf v$.
2. It has non-zero solutions iff $\det(A - \lambda I_n) = 0$. Write out the determinant

$$|A - \lambda I_n| = \begin{vmatrix} a_{11}-\lambda & a_{12} & \cdots & a_{1n}\\ a_{21} & a_{22}-\lambda & \cdots & a_{2n}\\
\vdots & \vdots & \ddots & \vdots \\ a_{n1} & a_{n2} & \cdots & a_{nn}-\lambda\end{vmatrix} = 0,$$

   *In words:* subtract $\lambda$ from every **diagonal** entry only (row $i$, column $i$); every off-diagonal entry
   $a_{ij}$ stays as it is. This determinant is a polynomial of degree $n$ in $\lambda$.
3. Find all its roots $\lambda_1, \lambda_2, \dots$ — the eigenvalues.
4. Plug each $\lambda_i$ back into step 1 and solve for the eigenvectors (Gauss–Jordan elimination, Module 09).

The next cell runs this procedure for a second matrix, $\begin{pmatrix}5&2\\2&1\end{pmatrix}$, and plots the
characteristic polynomials of three matrices so that you can *see* their roots.

In [ ]:
# ── 2.2 The procedure as a function, and the characteristic polynomials as curves ──
def eigen_by_hand(A_rows):
    """Run the four-step procedure exactly with SymPy and print each step; returns {eigenvalue: [eigenvectors]}."""
    A_ = sp.Matrix(A_rows)
    n_ = A_.shape[0]
    f_ = sp.expand((A_ - lam * sp.eye(n_)).det())
    print(f"A = {A_.tolist()}\n  step 2: f(λ) = det(A − λI) = {f_} = {sp.factor(f_)}")
    result = {}
    for r in sp.solve(f_, lam):
        vecs = [sp.simplify(b) for b in (A_ - r * sp.eye(n_)).nullspace()]
        result[r] = vecs
        print(f"  step 3: λ = {r} (≈ {float(r):.4f});  step 4: eigenvector(s) {[list(b) for b in vecs]}")
    return result


eig_52 = eigen_by_hand([[5, 2], [2, 1]])

grid = np.linspace(-3, 7, 400)
curves = {"[[3,4],[2,1]]": grid**2 - 4 * grid - 5,
          "[[5,2],[2,1]]": grid**2 - 6 * grid + 1,
          "rotation by 30°": grid**2 - np.sqrt(3) * grid + 1}
fig, ax = plt.subplots(figsize=(6.4, 3.6))
for (name, fvals), c in zip(curves.items(), ["tab:blue", "tab:orange", "tab:green"]):
    ax.plot(grid, fvals, color=c, label=f"f(λ) of {name}")
for r, c in [(-1, "tab:blue"), (5, "tab:blue"), (3 - 2 * np.sqrt(2), "tab:orange"), (3 + 2 * np.sqrt(2), "tab:orange")]:
    ax.plot(r, 0, "o", color=c)
ax.axhline(0, color="0.4", lw=0.8)
ax.set_ylim(-12, 15); ax.set_xlabel("λ"); ax.set_ylabel("f(λ) = det(A − λI)")
ax.set_title("eigenvalues = where the characteristic polynomial crosses zero")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

🔍 **What just happened?** For $A = \begin{pmatrix}5&2\\2&1\end{pmatrix}$:

$$f(\lambda) = \det\begin{pmatrix}5-\lambda & 2\\ 2 & 1-\lambda\end{pmatrix} = (5-\lambda)(1-\lambda) - 2\cdot2
= \lambda^2 - 6\lambda + 1 .$$

*In words:* the trace is 6 and the determinant $5 - 4 = 1$, so $f(\lambda) = \lambda^2 - 6\lambda + 1$, exactly the
2 × 2 formula. This polynomial does not factor over the integers; the quadratic formula gives
$\lambda = \tfrac{6\pm\sqrt{36-4}}{2} = 3\pm2\sqrt2 \approx 0.1716$ and $5.8284$, which SymPy printed together with
the eigenvectors $(1\pm\sqrt2,\ 1)$ — eigenvalues and eigenvectors need not be whole numbers.

In the plot each characteristic polynomial is a parabola; its **zero crossings are the eigenvalues** (dots): $-1$ and
5 for the matrix of Section 2.1, $0.17$ and $5.83$ for the second matrix. The green parabola — the polynomial
$\lambda^2 - \sqrt3\lambda + 1$ of a rotation by 30° — never reaches zero: a rotation has **no real eigenvalue**,
because it turns every direction. Section 4 shows what happens instead.

✏️ **Your turn.** By hand, find the characteristic polynomial, eigenvalues and eigenvectors of
$\begin{pmatrix}2&2\\-4&8\end{pmatrix}$ (from Section 1) and of $\begin{pmatrix}4&1\\2&3\end{pmatrix}$; check both
with `eigen_by_hand`. (Answers in the ✅ Checkpoint.)

### 📘 Lecture notes — the characteristic polynomial and the general procedure

**Derivation.** $A\mathbf x = \lambda\mathbf x \Leftrightarrow A\mathbf x = \lambda I_n\mathbf x \Leftrightarrow
(A-\lambda I_n)\mathbf x = \mathbf 0$: for each fixed $\lambda$ a homogeneous linear system on $n$ unknowns. It has a
non-trivial solution **iff $A-\lambda I_n$ is singular** (not invertible) **iff $\det(A-\lambda I_n) = 0$**. (It is *not*
true that $A - \lambda I_n$ must be the zero matrix.)

**Definitions.** Characteristic polynomial $f(\lambda) = \det(A - \lambda I_n)$ (degree $n$); characteristic equation
$\det(A - \lambda I_n) = 0$, also written $|A-\lambda I_n| = 0$. **Theorem:** $\lambda_0$ is an eigenvalue of $A$ iff
$f(\lambda_0) = 0$. Finding eigenvalues is a *nonlinear* problem (roots of a polynomial); finding the eigenvectors
afterwards is a *linear* one.

**Procedure.** (1) Write $A - \lambda I_n$ (subtract $\lambda$ on the diagonal only). (2) Compute its determinant.
(3) Find all roots. (4) For each root solve $(A-\lambda I_n)\mathbf v = \mathbf 0$; any non-zero solution is an
eigenvector.

**2 × 2 shortcut.** $f(\lambda) = \lambda^2 - \operatorname{tr}(A)\lambda + \det A$.

**Worked examples.**
* $A = \begin{pmatrix}3&4\\2&1\end{pmatrix}$: $(3-\lambda)(1-\lambda) - 8 = \lambda^2 - 4\lambda - 5 = (\lambda+1)(\lambda-5)$;
  $\lambda=5$: $\begin{pmatrix}-2&4\\2&-4\end{pmatrix}\mathbf x = \mathbf 0 \Rightarrow \mathbf x = (2,1)$;
  $\lambda=-1$: $\begin{pmatrix}4&4\\2&2\end{pmatrix}\mathbf x = \mathbf 0 \Rightarrow \mathbf x = (1,-1)$.
* $A = \begin{pmatrix}5&2\\2&1\end{pmatrix}$: $f(\lambda) = (5-\lambda)(1-\lambda) - 2\cdot2 = \lambda^2 - 6\lambda + 1$,
  roots $3\pm2\sqrt2$.

**Common confusions.** Subtract $\lambda$ only on the diagonal (in row 2 the entries are $a_{21}$ and $a_{22}-\lambda$,
not $a_{11}$ and $a_{12}-\lambda$). The zero vector solves every $(A-\lambda I)\mathbf x = \mathbf 0$ and is never an
eigenvector. A $2\times2$ matrix has at most 2 eigenvalues, an $n\times n$ matrix at most $n$.

> 🎓 **Exam tip.** After finding the roots, check them twice for free: their **sum must equal the trace** and their
> **product the determinant** ($5 + (-1) = 4 = 3 + 1$ and $5\cdot(-1) = -5 = 3 - 8$). Then verify one eigenvector by
> multiplying.

---
## 3 · A larger example: a 3 × 3 matrix by cofactor expansion

> 📍 **Where we are.** The recipe is the same in three dimensions, but the determinant now needs a **cofactor
> expansion** (Module 08), the polynomial is a **cubic**, and each eigenvector comes from a 3 × 3 homogeneous system.
> This is the size of matrix you will be asked to handle by hand.

🧮 **Math refresher — cofactor (Laplace) expansion.** For an $n\times n$ matrix $A$, the determinant can be expanded
along any row $i$ or any column $j$:

$$\det(A) = \sum_{j=1}^{n}(-1)^{i+j}\,a_{ij}\,\det(M_{ij}) \quad\text{(row } i\text{)},\qquad
\det(A) = \sum_{i=1}^{n}(-1)^{i+j}\,a_{ij}\,\det(M_{ij}) \quad\text{(column } j\text{)}.$$

*In words:* walk along one row (or column); multiply each entry $a_{ij}$ (row $i$, column $j$) by the determinant of
the **minor** $M_{ij}$ — the smaller matrix left after deleting row $i$ **and** column $j$ — and by the sign factor
$(-1)^{i+j}$, then add everything up. The signs follow a chessboard: $\begin{pmatrix}+&-&+\\-&+&-\\+&-&+\end{pmatrix}$.
Choose the row or column with the most zeros (or the simplest entries) to save work.

**Example.** Find the eigenelements of $B = \begin{pmatrix}1&1&-1\\2&3&-4\\4&1&-4\end{pmatrix}$. The characteristic
equation is

$$\det(B-\lambda I_3) = \begin{vmatrix}1-\lambda & 1 & -1\\ 2 & 3-\lambda & -4\\ 4 & 1 & -4-\lambda\end{vmatrix} = 0 .$$

*In words:* subtract $\lambda$ from the three diagonal entries of $B$ and ask for which $\lambda$ the determinant vanishes.

We expand along **column 2**, because its entries $1,\ 3-\lambda,\ 1$ are the simplest. Column 2 has the signs
$-,\ +,\ -$ (positions (1,2), (2,2), (3,2) of the chessboard):

$$\det(B-\lambda I_3) = -\,1\cdot\begin{vmatrix}2 & -4\\ 4 & -4-\lambda\end{vmatrix}
\;+\;(3-\lambda)\begin{vmatrix}1-\lambda & -1\\ 4 & -4-\lambda\end{vmatrix}
\;-\;1\cdot\begin{vmatrix}1-\lambda & -1\\ 2 & -4\end{vmatrix}.$$

*In words:* each term is (sign) × (entry of column 2) × (determinant of what is left when we cross out that entry's
row and column 2).

The three 2 × 2 minors, one line each:

$$\begin{aligned}
\det M_{12} &= 2(-4-\lambda) - (-4)(4) = -8 - 2\lambda + 16 = 8 - 2\lambda,\\
\det M_{22} &= (1-\lambda)(-4-\lambda) - (-1)(4) = (-4 - \lambda + 4\lambda + \lambda^2) + 4 = \lambda^2 + 3\lambda,\\
\det M_{32} &= (1-\lambda)(-4) - (-1)(2) = -4 + 4\lambda + 2 = 4\lambda - 2 .
\end{aligned}$$

*In words:* each minor is a 2 × 2 determinant "$ad - bc$" of the four entries left after crossing out one row and
column 2.

Putting them together and expanding $(3-\lambda)(\lambda^2+3\lambda) = 3\lambda^2 + 9\lambda - \lambda^3 - 3\lambda^2 =
9\lambda - \lambda^3$:

$$\det(B-\lambda I_3) = -(8 - 2\lambda) + (9\lambda - \lambda^3) - (4\lambda - 2) = -\lambda^3 + 7\lambda - 6
= -(\lambda^3 - 7\lambda + 6).$$

*In words:* the characteristic polynomial is $-\lambda^3 + 7\lambda - 6$; setting it to zero is the same as solving
$\lambda^3 - 7\lambda + 6 = 0$.

⚠️ **Common confusion — the overall sign.** For an $n\times n$ matrix the highest term of $\det(A - \lambda I_n)$ is
$(-\lambda)^n$, so for $n = 3$ the polynomial starts with $-\lambda^3$. Many books (and SymPy's `charpoly`) use
$\det(\lambda I_n - A) = (-1)^n\det(A - \lambda I_n)$ instead, which always starts with $+\lambda^n$. If you use the
signs $+,-,+$ for column 2 by mistake, you also get $\lambda^3 - 7\lambda + 6$. None of this changes the **roots**:
$f(\lambda) = 0$ and $-f(\lambda) = 0$ have the same solutions. It matters only when you report the polynomial itself.

In [ ]:
# ── 3.1 The column-2 cofactor expansion, term by term (exact, with SymPy) ──
B_sym = sp.Matrix([[1, 1, -1], [2, 3, -4], [4, 1, -4]])
BL = B_sym - lam * sp.eye(3)

total = 0
for i in range(3):                                          # rows 1..3 of column 2 (index j = 1)
    minor = BL.copy()
    minor.row_del(i)
    minor.col_del(1)
    sign = (-1) ** (i + 1 + 2)                              # (−1)^(i+j) with 1-based i and j = 2
    term = sign * BL[i, 1] * minor.det()
    total += term
    print(f"row {i + 1}: sign {sign:+d} × entry ({BL[i, 1]}) × det{minor.tolist()} = {sp.expand(term)}")

print("\nsum of the three terms      :", sp.expand(total))
print("det(B − λI) computed directly:", sp.expand(BL.det()))
print("SymPy charpoly = det(λI − B) :", B_sym.charpoly(lam).as_expr())
print("factored                     :", sp.factor(BL.det()))

🔍 **What just happened?** The three printed terms are exactly the three pieces of the hand calculation:
$-(8-2\lambda) = 2\lambda - 8$, $(3-\lambda)(\lambda^2+3\lambda) = -\lambda^3 + 9\lambda$ and $-(4\lambda - 2) =
2 - 4\lambda$. Their sum, $-\lambda^3 + 7\lambda - 6$, agrees with SymPy's direct determinant, while SymPy's
`charpoly` returns the monic version $\lambda^3 - 7\lambda + 6 = \det(\lambda I - B)$ — the sign convention of the
warning above. Factored, $\det(B - \lambda I) = -(\lambda-1)(\lambda-2)(\lambda+3)$.

🧮 **Math refresher — factoring a cubic by hand.** If a polynomial with integer coefficients and leading coefficient 1
has an integer root, that root divides the constant term (here 6). So try $\pm1, \pm2, \pm3, \pm6$:
$1^3 - 7\cdot1 + 6 = 0$ ✓. Then divide out the factor $(\lambda - 1)$:
$\lambda^3 - 7\lambda + 6 = (\lambda - 1)(\lambda^2 + \lambda - 6)$ (multiply back to check:
$\lambda^3 + \lambda^2 - 6\lambda - \lambda^2 - \lambda + 6$), and factor the quadratic:
$\lambda^2 + \lambda - 6 = (\lambda - 2)(\lambda + 3)$. So

$$\lambda^3 - 7\lambda + 6 = (\lambda - 1)(\lambda - 2)(\lambda + 3) = 0 \quad\Rightarrow\quad \lambda = 1,\ 2,\ -3 .$$

*In words:* the three eigenvalues of $B$ are 1, 2 and $-3$. Check: $1 + 2 - 3 = 0 = \operatorname{tr}B = 1 + 3 - 4$ and
$1\cdot2\cdot(-3) = -6 = \det B$.

**The eigenvectors.** For $\lambda = 1$ we solve $(B - I_3)\mathbf x = \mathbf 0$:

$$\begin{pmatrix}0&1&-1\\2&2&-4\\4&1&-5\end{pmatrix}\begin{pmatrix}x_1\\x_2\\x_3\end{pmatrix} = \mathbf 0
\quad\Longleftrightarrow\quad
\begin{cases} x_2 - x_3 = 0\\ 2x_1 + 2x_2 - 4x_3 = 0\\ 4x_1 + x_2 - 5x_3 = 0\end{cases}$$

*In words:* each row of the matrix times $\mathbf x$ gives one equation. The first says $x_2 = x_3$; substituting into
the second gives $2x_1 - 2x_3 = 0$, i.e. $x_1 = x_3$; the third is then automatic ($4x_3 + x_3 - 5x_3 = 0$). So
$x_1 = x_2 = x_3$ and $\mathbf x = (1,1,1)$. The next cell does the same for all three eigenvalues by row reduction.

In [ ]:
# ── 3.2 All three eigenvectors of B by row reduction (RREF), checked by multiplication ──
for r in [1, 2, -3]:
    M = B_sym - r * sp.eye(3)
    R, pivots = M.rref()                                    # Gauss–Jordan elimination, exact fractions
    v = M.nullspace()[0]
    v = v / sp.gcd(list(v))                                 # simplest integer representative
    print(f"λ = {r:2d}:  B − λI = {M.tolist()}")
    print(f"         RREF = {R.tolist()}  (pivot columns {[p + 1 for p in pivots]}, x3 free)")
    print(f"         eigenvector {list(v)};  B·v = {list(B_sym * v)} = {r}·{list(v)}\n")

w_np, V_np = np.linalg.eig(np.array([[1, 1, -1], [2, 3, -4], [4, 1, -4]], dtype=float))
w_np, V_np = w_np.real, V_np.real                          # all imaginary parts are 0 here
print("NumPy eigenvalues:", w_np.round(6))
print("NumPy eigenvectors (columns, length 1), rescaled so that the first entry is 1:")
print(format_matrix(V_np / V_np[0]))

🔍 **What just happened?** Row reduction of $B - \lambda I$ leaves two pivots and one free variable for each eigenvalue,
so each eigenspace is a line:

* $\lambda = 1$: RREF $\begin{pmatrix}1&0&-1\\0&1&-1\\0&0&0\end{pmatrix}$ means $x_1 = x_3$, $x_2 = x_3$ → $(1,1,1)$.
* $\lambda = 2$: RREF $\begin{pmatrix}1&0&-1\\0&1&-2\\0&0&0\end{pmatrix}$ → $x_1 = x_3$, $x_2 = 2x_3$ → $(1,2,1)$.
* $\lambda = -3$: RREF $\begin{pmatrix}1&0&-1/11\\0&1&-7/11\\0&0&0\end{pmatrix}$ → $x_1 = x_3/11$, $x_2 = 7x_3/11$; choose
  $x_3 = 11$ → $(1, 7, 11)$.

Each is confirmed by multiplying: e.g. $B(1,7,11)^\top = (1+7-11,\ 2+21-44,\ 4+7-44) = (-3,-21,-33) = -3\cdot(1,7,11)$.
NumPy returns the same eigenvalues in its own order ($-3, 1, 2$ here — `eig` promises no particular order) and
eigenvectors scaled to length 1; dividing each column by its first entry recovers our integer vectors (columns
$(1,7,11)$, $(1,1,1)$, $(1,2,1)$).

### 3.2 A second 3 × 3 example: the population matrix

The Leslie matrix of Section 0 is a 3 × 3 matrix too. Its third column $(8, 0, -\lambda)$ (after subtracting
$\lambda I$) contains a zero, so we expand along **column 3** (signs $+, -, +$):

$$f(\lambda) = \det\begin{pmatrix}-\lambda & 6 & 8\\ \tfrac12 & -\lambda & 0\\ 0 & \tfrac12 & -\lambda\end{pmatrix}
= 8\begin{vmatrix}\tfrac12 & -\lambda\\ 0 & \tfrac12\end{vmatrix} - 0 + (-\lambda)\begin{vmatrix}-\lambda & 6\\ \tfrac12 & -\lambda\end{vmatrix}
= 8\left(\tfrac14 - 0\cdot(-\lambda)\right) - \lambda\left(\lambda^2 - 6\cdot\tfrac12\right) = -\lambda^3 + 3\lambda + 2 .$$

*In words:* the zero in the middle saves one minor; the first minor is $\tfrac12\cdot\tfrac12 - (-\lambda)\cdot0 =
\tfrac14$, the last one $\lambda^2 - 3$, and together they give $2 - \lambda^3 + 3\lambda$.

In [ ]:
# ── 3.3 The population matrix: characteristic polynomial, roots and eigenvectors (exact) ──
L_sym = sp.Matrix([[0, 6, 8], [sp.Rational(1, 2), 0, 0], [0, sp.Rational(1, 2), 0]])
f_L = sp.expand((L_sym - lam * sp.eye(3)).det())
print("f(λ) =", f_L, "=", sp.factor(f_L))
for value, alg_mult, vectors in L_sym.eigenvects():
    vecs = [list(v / sp.gcd(list(v))) for v in vectors]
    print(f"λ = {str(value):>2}: root of multiplicity {alg_mult}, independent eigenvectors: {vecs}")

🔍 **What just happened?** $f(\lambda) = -\lambda^3 + 3\lambda + 2 = -(\lambda - 2)(\lambda + 1)^2$. The population
matrix has the eigenvalue **2** — the long-run growth factor we saw in Section 0 — with eigenvector $(16, 4, 1)$, the
age structure we verified in Section 1. The other root, $-1$, appears **twice** (it is a *double root*: the factor
$(\lambda+1)$ is squared), yet it has only **one** independent eigenvector, $(4, -2, 1)$. A root that repeats but brings
fewer eigenvectors than its multiplicity is the first sign of a matrix that cannot be "diagonalised" — a story for
[Module 13 · 🧩 Diagonalization and the singular value decomposition](13_diagonalization_svd.ipynb). The negative entry of $(4,-2,1)$ also tells us that this eigenvector is not a possible population; it describes
a *deviation* that flips sign every year (eigenvalue $-1$) and that Section 9 shows is outgrown by the factor-2 growth.

✏️ **Your turn.** (a) Expand $\det(B - \lambda I_3)$ along **row 1** instead of column 2 and check that you get the same
polynomial. (b) Solve $(B - 2I)\mathbf x = \mathbf 0$ and $(B+3I)\mathbf x = \mathbf 0$ by hand (eliminate one unknown at a
time) and compare with the RREF printout. (c) What is $\det L$? Use $f(0)$.

### 📘 Lecture notes — a 3 × 3 example by cofactor expansion

**Reminder.** $\det(A) = \sum_{j}(-1)^{i+j}a_{ij}\det(M_{ij})$ (along row $i$) $= \sum_i(-1)^{i+j}a_{ij}\det(M_{ij})$
(along column $j$); $M_{ij}$ is the **minor** obtained by deleting row $i$ and column $j$; $(-1)^{i+j}$ is the sign
factor (alternating, chessboard pattern).

**Worked example.** $B = \begin{pmatrix}1&1&-1\\2&3&-4\\4&1&-4\end{pmatrix}$; expand $\det(B-\lambda I)$ along column 2
(it has the most 1's):

$$-1\cdot\begin{vmatrix}2&-4\\4&-4-\lambda\end{vmatrix} + (3-\lambda)\begin{vmatrix}1-\lambda&-1\\4&-4-\lambda\end{vmatrix}
- 1\cdot\begin{vmatrix}1-\lambda&-1\\2&-4\end{vmatrix} = -(8-2\lambda) + (3-\lambda)(\lambda^2+3\lambda) - (4\lambda-2)
= -\lambda^3 + 7\lambda - 6 .$$

*In words:* the characteristic equation is $\lambda^3 - 7\lambda + 6 = (\lambda-1)(\lambda-2)(\lambda+3) = 0$, so
$\lambda = 1, 2, -3$.

**Eigenvectors.** $\lambda = 1$: $(B - I)\mathbf x = \mathbf 0$ reads $x_2 - x_3 = 0$, $2x_1 + 2x_2 - 4x_3 = 0$,
$4x_1 + x_2 - 5x_3 = 0$ $\Rightarrow x_1 = x_2 = x_3$, $\mathbf x = (1,1,1)$. Likewise $\lambda = 2 \Rightarrow (1,2,1)$ and
$\lambda = -3 \Rightarrow (1,7,11)$.

**Second example.** $L = \begin{pmatrix}0&6&8\\\frac12&0&0\\0&\frac12&0\end{pmatrix}$, expanding along column 3:
$f(\lambda) = 8\left(\tfrac14 - 0\cdot(-\lambda)\right) - \lambda\left(\lambda^2 - 6\cdot\tfrac12\right) = -\lambda^3 + 3\lambda + 2
= -(\lambda-2)(\lambda+1)^2$.

**Common confusions.** For $n = 3$, $\det(A - \lambda I)$ starts with $-\lambda^3$; $\det(\lambda I - A)$ starts with
$+\lambda^3$ — same roots. Use the correct chessboard signs for the row or column you expand along (column 2 is
$-,+,-$). Two equations of $(B-\lambda I)\mathbf x = \mathbf 0$ are enough — the third is a combination of them, because
the matrix is singular.

> 🎓 **Exam tip.** Expand along the row or column with the most zeros, factor the cubic by trying divisors of the
> constant term, and check with $\sum\lambda_i = \operatorname{tr}$ and $\prod\lambda_i = \det$. Report eigenvectors
> as simple integer vectors and verify one by multiplication.

---
## 4 · Eigenvectors in pictures: five maps of the plane

> 📍 **Where we are.** We can now compute eigenelements; this section builds the intuition. We apply five 2 × 2
> matrices to a square cloud of 400 colour-coded points and draw each matrix's eigenvectors, stretched by their
> eigenvalues. Each map illustrates one situation you must recognise: two real eigenvalues, a repeated eigenvalue with
> a single eigen-direction, complex eigenvalues, a zero eigenvalue, and a symmetric matrix.

The five matrices (a classic overview from Deisenroth, Faisal & Ong, *Mathematics for Machine Learning*):

$$A_1 = \begin{pmatrix}\tfrac12 & 0\\ 0 & 2\end{pmatrix},\quad
A_2 = \begin{pmatrix}1 & \tfrac12\\ 0 & 1\end{pmatrix},\quad
A_3 = \begin{pmatrix}\cos\frac\pi6 & -\sin\frac\pi6\\ \sin\frac\pi6 & \cos\frac\pi6\end{pmatrix},\quad
A_4 = \begin{pmatrix}1 & -1\\ -1 & 1\end{pmatrix},\quad
A_5 = \begin{pmatrix}1 & \tfrac12\\ \tfrac12 & 1\end{pmatrix}.$$

*In words:* $A_1$ squashes the horizontal axis and stretches the vertical one; $A_2$ is a **shear** (it shifts points
sideways in proportion to their height); $A_3$ **rotates** by $\pi/6 = 30°$; $A_4$ collapses the plane onto a line;
$A_5$ is a symmetric stretch along the diagonals.

🧮 **Math refresher — complex numbers.** There is no real number whose square is $-1$; mathematicians simply introduce
one, $i$ with $i^2 = -1$. A **complex number** $a + bi$ has a real part $a$ and an imaginary part $b$ (Python writes
`0.87+0.5j`). Every quadratic has two roots once complex numbers are allowed: for the rotation,
$f(\lambda) = \lambda^2 - \sqrt3\,\lambda + 1$ has $p^2 - 4q = 3 - 4 = -1 < 0$, so
$\lambda = \tfrac{\sqrt3\pm\sqrt{-1}}{2} = \tfrac{\sqrt3}{2}\pm\tfrac12 i \approx 0.87\pm0.5i$ — exactly
$\cos\frac\pi6 \pm i\sin\frac\pi6$. Complex eigenvalues of a real matrix always come in such conjugate pairs
$a\pm bi$, and they signal **rotation** (in some plane); for a 2 × 2 matrix it means that no real direction is left
unturned.

In [ ]:
# ── 4.1 Eigenvalues, eigenvectors and determinants of the five maps ──
c30, s30 = np.cos(np.pi / 6), np.sin(np.pi / 6)
MAPS = {"A1 (stretch/squash)": np.array([[0.5, 0.0], [0.0, 2.0]]),
        "A2 (shear)": np.array([[1.0, 0.5], [0.0, 1.0]]),
        "A3 (rotation 30°)": np.array([[c30, -s30], [s30, c30]]),
        "A4 (collapse)": np.array([[1.0, -1.0], [-1.0, 1.0]]),
        "A5 (symmetric)": np.array([[1.0, 0.5], [0.5, 1.0]])}
for name, M in MAPS.items():
    vals, vecs = np.linalg.eig(M)
    rank_vecs = np.linalg.matrix_rank(vecs, tol=1e-8)
    print(f"{name:20s} λ = {np.round(vals, 3)}   det = {np.linalg.det(M):.2f}   "
          f"independent eigenvectors: {rank_vecs}")

In [ ]:
# ── 4.2 The figure: 400 colour-coded points and their images A·x, eigenvectors scaled by λ ──
g = np.linspace(-1, 1, 20)
gx, gy = np.meshgrid(g, g)
pts = np.vstack([gx.ravel(), gy.ravel()])                  # 2 × 400: one column per point
colours = plt.cm.viridis((pts[1] + 1) / 2)                  # colour = original height (blue bottom → yellow top)

fig, axes = plt.subplots(2, 3, figsize=(10.5, 7.2))
ax = axes[0, 0]
ax.scatter(pts[0], pts[1], c=colours, s=6)
ax.set_title("original points x (400)")
for ax, (name, M) in zip(axes.ravel()[1:], MAPS.items()):
    img = M @ pts
    ax.scatter(img[0], img[1], c=colours, s=6)
    vals, vecs = np.linalg.eig(M)
    if np.iscomplexobj(vals) and np.abs(vals.imag).max() > 1e-12:
        ax.text(0.03, 0.04, "complex λ: no real\neigen-direction", transform=ax.transAxes, fontsize=8, color="tab:red")
    else:
        drawn = []
        for k in range(2):
            unit = vecs[:, k].real
            if any(abs(unit[0] * d[1] - unit[1] * d[0]) < 1e-6 for d in drawn):
                continue                                         # parallel to one already drawn (shear): skip
            drawn.append(unit)
            vec = vals[k].real * unit                            # eigenvector stretched by its eigenvalue
            if np.linalg.norm(vec) > 1e-9:
                ax.annotate("", xy=vec, xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color="tab:red", lw=2))
            else:                                                # eigenvalue 0: this direction is squashed to 0
                ax.plot([-1.6 * unit[0], 1.6 * unit[0]], [-1.6 * unit[1], 1.6 * unit[1]], ":", color="tab:red")
                ax.text(0.1, 1.5, "λ = 0 direction\n(sent to 0)", color="tab:red", fontsize=7)
    lam_txt = ", ".join(f"{v.real:.2f}" if abs(v.imag) < 1e-12 else f"{v.real:.2f}{v.imag:+.2f}i" for v in vals)
    ax.set_title(f"{name}\nλ = {lam_txt};  det = {np.linalg.det(M):.2f}", fontsize=9)
for ax in axes.ravel():
    ax.axhline(0, color="0.6", lw=0.6); ax.axvline(0, color="0.6", lw=0.6)
    ax.set_xlim(-2.3, 2.3); ax.set_ylim(-2.3, 2.3); ax.set_aspect("equal")
    ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$")
fig.suptitle("Five linear maps applied to the same 400 points (colour = original height); red arrows = λ·eigenvector",
             fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Read each panel with the printed table:

* **$A_1$** ($\lambda = 0.5, 2$; det 1): the eigenvectors are the coordinate axes. The cloud is squashed to half its
  width and stretched to twice its height; the area is unchanged because $\det A_1 = 0.5\times2 = 1$.
* **$A_2$, the shear** ($\lambda = 1, 1$; det 1): the eigenvalue 1 is *repeated*, yet the eigenvector matrix has rank 1
  — there is only **one** eigen-direction, the horizontal axis (NumPy returns two numerically parallel columns; the
  figure draws the direction once). Points on that axis do not move; every other point moves sideways, so every
  other direction is turned.
* **$A_3$, the rotation** ($\lambda = 0.87\pm0.50i$; det 1): no real eigenvector at all — every arrow is turned by 30°.
  The eigenvalues are complex with $|\lambda| = \sqrt{0.87^2 + 0.5^2} = 1$: a rotation neither stretches nor shrinks.
* **$A_4$, the collapse** ($\lambda = 2, 0$; det 0): the eigenvector $(1,-1)/\sqrt2$ is doubled, while the eigenvector
  $(1,1)/\sqrt2$ is sent to $\mathbf 0$ (red dotted line: eigenvalue 0). The whole square collapses onto the line $x_2 = -x_1$:
  a zero eigenvalue means the matrix is **singular** (det 0) and loses a dimension (rank 1 = one non-zero eigenvalue).
* **$A_5$, symmetric** ($\lambda = 1.5, 0.5$; det 0.75): the two eigen-directions are the diagonals, and they are
  **perpendicular** — a property of every symmetric matrix (Section 7). The square becomes a rhombus stretched by 1.5
  along $(1,1)$ and squashed by 0.5 along $(1,-1)$; area factor $1.5\times0.5 = 0.75 = \det A_5$.

In all five panels the determinant equals the product of the eigenvalues (for $A_3$: $(0.87+0.5i)(0.87-0.5i) =
0.75 + 0.25 = 1$). Section 5 proves it.

✏️ **Your turn.** Add two maps to the dictionary and rerun: the rotation by 90°,
$\begin{pmatrix}0&-1\\1&0\end{pmatrix}$, and the reflection in the horizontal axis, $\begin{pmatrix}1&0\\0&-1\end{pmatrix}$.
Predict the eigenvalues before you run (characteristic polynomials $\lambda^2 + 1$ and $(1-\lambda)(-1-\lambda)$).

---
## 5 · Properties: independence, determinant, trace, rank, inverses and powers

> 📍 **Where we are.** A handful of facts about eigenvalues make hand calculations checkable and explain the
> long-run behaviour of matrix powers. Each comes with a short proof — they all follow from $A\mathbf x = \lambda\mathbf x$
> and the characteristic polynomial.

### 5.1 Eigenvectors with distinct eigenvalues are linearly independent

**Theorem.** If $\mathbf x_1, \dots, \mathbf x_n$ are eigenvectors of $A\in\mathbb R^{n\times n}$ with $n$ **distinct**
eigenvalues $\lambda_1,\dots,\lambda_n$ (no two equal), then they are linearly independent — so they form a **basis**
of $\mathbb R^n$ (Module 11).

**Proof for two eigenvectors.** Suppose $c_1\mathbf x_1 + c_2\mathbf x_2 = \mathbf 0$ (★). We must show
$c_1 = c_2 = 0$.

$$\begin{aligned}
\text{multiply (★) by } A: &\quad c_1\lambda_1\mathbf x_1 + c_2\lambda_2\mathbf x_2 = \mathbf 0\\
\text{multiply (★) by } \lambda_2: &\quad c_1\lambda_2\mathbf x_1 + c_2\lambda_2\mathbf x_2 = \mathbf 0\\
\text{subtract:} &\quad c_1(\lambda_1 - \lambda_2)\,\mathbf x_1 = \mathbf 0 .
\end{aligned}$$

*In words:* applying $A$ multiplies each eigenvector by its own eigenvalue; subtracting the two lines cancels
$\mathbf x_2$. Since $\lambda_1 \neq \lambda_2$ and $\mathbf x_1 \neq \mathbf 0$, we must have $c_1 = 0$; then (★) says
$c_2\mathbf x_2 = \mathbf 0$, so $c_2 = 0$. ∎ (For $n$ eigenvectors the same argument works by induction: assume the
first $k$ are independent and show that adding the $(k+1)$-th keeps them independent.)

**Repeated eigenvalues.** If an eigenvalue repeats, the theorem says nothing: there may still be enough independent
eigenvectors ($I_2$ has $\lambda = 1, 1$ and *every* vector is an eigenvector; the matrix $D$ of Section 6 has
$\lambda = 1$ twice with a whole plane of eigenvectors), or too few (the shear $A_2$, and the eigenvalue $-1$ of the
population matrix, have only one eigen-direction each). Independence then has to be checked, not assumed.

In [ ]:
# ── 5.1 Rank of the eigenvector matrix: distinct vs repeated eigenvalues ──
cases = {"B (λ = 1, 2, −3)": np.array([[1, 1, -1], [2, 3, -4], [4, 1, -4]], dtype=float),
         "identity I2 (λ = 1, 1)": np.eye(2),
         "shear A2 (λ = 1, 1)": MAPS["A2 (shear)"],
         "Leslie L (λ = 2, −1, −1)": L,
         "rotation A3 (complex)": MAPS["A3 (rotation 30°)"]}
for name, M in cases.items():
    vals, vecs = np.linalg.eig(M)
    print(f"{name:26s} n = {M.shape[0]}, rank of eigenvector matrix = {np.linalg.matrix_rank(vecs, tol=1e-6)}")

🔍 **What just happened?** With three distinct eigenvalues, $B$'s three eigenvectors have rank 3 — a basis of
$\mathbb R^3$, as the theorem promises. The identity has a repeated eigenvalue but still two independent eigenvectors
(rank 2). The shear and the population matrix fall short: rank 1 instead of 2, and rank 2 instead of 3 — the repeated
eigenvalue brought only one direction. The rotation has rank 2, but its eigenvectors are *complex* vectors: there is
still no real eigen-direction. "Distinct eigenvalues ⇒ independent eigenvectors" is a one-way street.

### 5.2 Determinant = product, trace = sum of the eigenvalues

Allowing complex roots, every polynomial of degree $n$ with leading term $(-\lambda)^n$ factors into $n$ linear factors,
so the characteristic polynomial can be written with its roots $\lambda_1,\dots,\lambda_n$ (repeated roots repeated):

$$f(\lambda) = \det(A - \lambda I_n) = (\lambda_1 - \lambda)(\lambda_2 - \lambda)\cdots(\lambda_n - \lambda).$$

*In words:* both sides are polynomials with the same roots and the same leading term, so they are equal.

Setting $\lambda = 0$ gives

$$\det A = f(0) = \lambda_1\lambda_2\cdots\lambda_n = \prod_{i=1}^n\lambda_i .$$

*In words:* the determinant is the **product of the eigenvalues** — the area/volume factor of the map is the product
of its stretch factors along the eigen-directions. ($\prod$, capital pi, means "multiply over $i$", as $\sum$ means
"add".)

Comparing the coefficients of $\lambda^{n-1}$ on both sides (for $n=2$: $\lambda^2 - (a+d)\lambda + (ad-bc) =
\lambda^2 - (\lambda_1+\lambda_2)\lambda + \lambda_1\lambda_2$) gives

$$\operatorname{tr}A = a_{11} + a_{22} + \dots + a_{nn} = \lambda_1 + \lambda_2 + \dots + \lambda_n = \sum_{i=1}^n\lambda_i .$$

*In words:* the **trace** (sum of the diagonal entries) equals the **sum of the eigenvalues**.

In [ ]:
# ── 5.2 det = Π λ and tr = Σ λ for every matrix of this module ──
all_mats = {"[[3,4],[2,1]]": A_notes.astype(float), "[[2,2],[-4,8]]": A_ex1.astype(float),
            "[[5,2],[2,1]]": np.array([[5.0, 2], [2, 1]]), "B (3×3)": cases["B (λ = 1, 2, −3)"],
            "Leslie L": L, **MAPS}
rows = []
for name, M in all_mats.items():
    vals = np.linalg.eigvals(M)
    rows.append(dict(matrix=name, eigenvalues=np.round(vals, 3), det=np.linalg.det(M), prod_lambda=np.prod(vals).real,
                     trace=np.trace(M), sum_lambda=np.sum(vals).real))
print(pd.DataFrame(rows).round(4).to_string(index=False))

🔍 **What just happened?** In every row the columns `det` and `prod_lambda` agree, and so do `trace` and
`sum_lambda`: $\det\begin{pmatrix}3&4\\2&1\end{pmatrix} = -5 = 5\cdot(-1)$; $B$ has trace 0 and determinant $-6$;
the population matrix has $\det L = 2$, the product $2\cdot(-1)\cdot(-1)$, and trace $0 = 2 - 1 - 1$. For the rotation
the eigenvalues are complex, but their product $(0.866+0.5i)(0.866-0.5i) = 1$ and their sum $1.732 = 2\cos 30°$ are
real, as they must be for a real matrix.

### 5.3 Rank and the number of non-zero eigenvalues

$\lambda = 0$ is an eigenvalue $\iff$ $A\mathbf x = \mathbf 0$ for some $\mathbf x\neq\mathbf 0$ $\iff$ $A$ is singular
$\iff$ $\det A = 0$ (consistent with $\det A = \prod\lambda_i$). One more entry for the invertible matrix theorem of
Module 09: **$A$ is invertible iff 0 is not an eigenvalue**.

A popular rule says "the rank of $A$ equals the number of non-zero eigenvalues". That is true **if $A$ is
diagonalizable** (has $n$ independent eigenvectors, [Module 13 · 🧩 Diagonalization and the singular value decomposition](13_diagonalization_svd.ipynb)), but not in general: the nilpotent matrix
$N = \begin{pmatrix}0&1\\0&0\end{pmatrix}$ has $f(\lambda) = \lambda^2$, so both eigenvalues are 0, yet its rank is 1. In
general, rank $\geq$ the number of non-zero eigenvalues (counted with multiplicity).

### 5.4 Eigenvalues of $A^{-1}$, $A^k$ and $A + cI$ — same eigenvectors

If $A\mathbf x = \lambda\mathbf x$:

$$A^{-1}\mathbf x = \tfrac1\lambda\mathbf x \ (A \text{ invertible}),\qquad A^k\mathbf x = \lambda^k\mathbf x,\qquad
(A + cI)\mathbf x = (\lambda + c)\mathbf x .$$

*In words:* the inverse has the reciprocal eigenvalues, the $k$-th power the $k$-th powers, a shift by $cI$ the shifted
eigenvalues — and **all keep the same eigenvectors**. Proofs, one line each: multiply $A\mathbf x = \lambda\mathbf x$ by
$A^{-1}$ to get $\mathbf x = \lambda A^{-1}\mathbf x$ and divide by $\lambda\neq0$; apply $A$ again,
$A^2\mathbf x = A(\lambda\mathbf x) = \lambda A\mathbf x = \lambda^2\mathbf x$, and repeat $k$ times; expand
$(A + cI)\mathbf x = A\mathbf x + c\mathbf x = (\lambda + c)\mathbf x$.

In [ ]:
# ── 5.3–5.4 Rank vs non-zero eigenvalues; eigenvalues of A⁻¹, A³ and A + 10 I ──
N = np.array([[0.0, 1.0], [0.0, 0.0]])
for name, M in [("A4 (collapse)", MAPS["A4 (collapse)"]), ("Leslie L", L), ("nilpotent N", N)]:
    vals = np.linalg.eigvals(M)
    print(f"{name:14s} rank = {np.linalg.matrix_rank(M)},  eigenvalues = {vals.round(3)},  "
          f"non-zero eigenvalues = {int(np.sum(np.abs(vals) > 1e-9))}")

A = A_notes.astype(float)
x5, x_1 = np.array([2.0, 1.0]), np.array([1.0, -1.0])      # eigenvectors for λ = 5 and λ = −1
for name, M in [("A", A), ("A⁻¹", np.linalg.inv(A)), ("A³", np.linalg.matrix_power(A, 3)), ("A + 10I", A + 10 * np.eye(2))]:
    print(f"{name:8s} eigenvalues {np.sort(np.linalg.eigvals(M)).round(4)};   "
          f"M·(2,1) = {(M @ x5).round(4)},  M·(1,−1) = {(M @ x_1).round(4)}")

🔍 **What just happened?** The collapse $A_4$ (rank 1, one non-zero eigenvalue) and the population matrix (rank 3,
three non-zero eigenvalues, counting the double root $-1$ twice) obey the rank rule (for $L$ it holds simply because $L$ is invertible, although $L$ is not diagonalizable); the nilpotent $N$ breaks it (rank 1, no non-zero eigenvalue), because
it is not diagonalizable — it has only one eigen-direction. For $A = \begin{pmatrix}3&4\\2&1\end{pmatrix}$ with
eigenvalues $-1$ and 5: $A^{-1}$ has $-1$ and $0.2 = 1/5$; $A^3$ has $(-1)^3 = -1$ and $5^3 = 125$; $A + 10I$ has 9 and 15.
And the same two vectors stay eigenvectors: $A^3(2,1) = (250, 125) = 125\cdot(2,1)$ and $A^{-1}(2,1) = (0.4, 0.2)$.
The rule for $A^k$ is the key to the population model (Section 9) and to matrix powers in [Module 13 · 🧩 Diagonalization and the singular value decomposition](13_diagonalization_svd.ipynb); the rule for $A + cI$
rescues power iteration in Section 8.

✏️ **Your turn.** (a) Prove that $A$ and $A^\top$ have the same characteristic polynomial (hint: $\det M^\top = \det M$
and $(A - \lambda I)^\top = A^\top - \lambda I$). Check numerically that $\begin{pmatrix}3&4\\2&1\end{pmatrix}^\top$ has
eigenvalues 5 and $-1$ but **different** eigenvectors. (b) If $A$ has eigenvalues 2 and $-\tfrac12$, what are the
eigenvalues of $A^{-1}$, $A^2$, $A - 2I$, and what is $\det A$?

### 📘 Lecture notes — properties of eigenvalues and eigenvectors

**Independence.** Eigenvectors $\mathbf x_1,\dots,\mathbf x_n$ of $A\in\mathbb R^{n\times n}$ with $n$ distinct eigenvalues
($\lambda_i\neq\lambda_j$ whenever $i\neq j$) are linearly independent and form a basis of $\mathbb R^n$. Proof for two:
from $c_1\mathbf x_1 + c_2\mathbf x_2 = \mathbf 0$, apply $A$ and subtract $\lambda_2$ times the original:
$c_1(\lambda_1-\lambda_2)\mathbf x_1 = \mathbf 0 \Rightarrow c_1 = 0 \Rightarrow c_2 = 0$. With **repeated** eigenvalues,
independent eigenvectors may exist (identity) but are not guaranteed (shear $\begin{pmatrix}1&\frac12\\0&1\end{pmatrix}$).

**Determinant and trace.**

$$|A| = \prod_{i=1}^n\lambda_i, \qquad \operatorname{tr}A = \sum_{i=1}^n\lambda_i .$$

*In words:* the determinant is the product and the trace the sum of all eigenvalues (with multiplicity, complex ones
included).

**Rank.** 0 is an eigenvalue iff $A$ is singular. For a **diagonalizable** $A$, rank = number of non-zero eigenvalues;
in general rank $\ge$ that number ($\begin{pmatrix}0&1\\0&0\end{pmatrix}$: rank 1, eigenvalues 0, 0).

**Functions of $A$.** If $A\mathbf x = \lambda\mathbf x$ then $A^{-1}\mathbf x = \frac1\lambda\mathbf x$ ($A$ nonsingular),
$A^k\mathbf x = \lambda^k\mathbf x$ and $(A + cI)\mathbf x = (\lambda + c)\mathbf x$ — same eigenvector. $A^\top$ has the same
eigenvalues as $A$ (not the same eigenvectors).

**Five maps of the plane.** Stretch $\mathrm{diag}(\frac12, 2)$: $\lambda = \frac12, 2$, det 1. Shear: $\lambda = 1, 1$,
one eigen-direction. Rotation by $\pi/6$: $\lambda = 0.87\pm0.5i$, no real eigenvector, det 1. $\begin{pmatrix}1&-1\\-1&1
\end{pmatrix}$: $\lambda = 2, 0$, det 0 (collapse onto a line). $\begin{pmatrix}1&\frac12\\\frac12&1\end{pmatrix}$:
$\lambda = 1.5, 0.5$, perpendicular eigenvectors, det 0.75.

**Common confusions.** "Repeated eigenvalue ⇒ dependent eigenvectors" (not necessarily); "real matrix ⇒ real
eigenvalues" (rotations); "rank = number of non-zero eigenvalues" (only for diagonalizable matrices).

> 🎓 **Exam tip.** Given eigenvalues, you can answer without any eigenvectors: $\det$ (product), trace (sum),
> invertibility (is 0 among them?), and the eigenvalues of $A^{-1}$, $A^k$, $A+cI$.

---
## 6 · Triangular matrices: eigenvalues on the diagonal

> 📍 **Where we are.** For most matrices the characteristic polynomial is hard work. For **triangular** matrices it is
> free — and Gaussian elimination turns any matrix into a triangular one (Module 09), which is how computers get
> determinants. This section also produces our first **two-dimensional eigenspace**.

**Definition.** $T\in\mathbb R^{n\times n}$ is **upper triangular** if all entries below the diagonal are 0
($T_{ij} = 0$ for $i > j$), **lower triangular** if all entries above the diagonal are 0 ($T_{ij} = 0$ for $i < j$),
and **diagonal** if both hold.

$$\begin{pmatrix}T_{11} & T_{12} & \cdots & T_{1n}\\ 0 & T_{22} & \cdots & T_{2n}\\ \vdots & & \ddots & \vdots\\ 0 & 0 & \cdots & T_{nn}\end{pmatrix}
\qquad\text{or}\qquad
\begin{pmatrix}T_{11} & 0 & \cdots & 0\\ T_{21} & T_{22} & \cdots & 0\\ \vdots & & \ddots & \vdots\\ T_{n1} & T_{n2} & \cdots & T_{nn}\end{pmatrix}.$$

*In words:* in an upper triangular matrix everything below the main diagonal is zero; in a lower triangular one,
everything above it.

**Two facts.**

$$|T| = \prod_{i=1}^n T_{ii}, \qquad \langle\lambda_1,\lambda_2,\dots,\lambda_n\rangle = \langle T_{11}, T_{22},\dots,T_{nn}\rangle .$$

*In words:* the determinant of a triangular matrix is the product of its diagonal entries, and its eigenvalues are
exactly its diagonal entries.

**Why.** Expand an upper triangular determinant along its first column: the only non-zero entry is $T_{11}$, and its
minor is again upper triangular, so $|T| = T_{11}\cdot|\text{minor}|$; repeating gives the product of the diagonal.
(Lower triangular: expand along the first row.) Now $T - \lambda I$ is triangular too, with diagonal entries
$T_{ii} - \lambda$, so

$$f(\lambda) = \det(T - \lambda I) = (T_{11}-\lambda)(T_{22}-\lambda)\cdots(T_{nn}-\lambda),$$

*In words:* the characteristic polynomial is already factored, and its zeros are $T_{11}, \dots, T_{nn}$.

In [ ]:
# ── 6.1 Triangular patterns, and the general 3 × 3 upper-triangular characteristic polynomial ──
fig, axes = plt.subplots(1, 3, figsize=(9, 3.0))
for ax, (title, keep) in zip(axes, [("upper triangular", lambda i, j: j >= i),
                                     ("lower triangular", lambda i, j: j <= i),
                                     ("diagonal", lambda i, j: i == j)]):
    for i in range(4):
        for j in range(4):
            on = keep(i, j)
            ax.add_patch(plt.Rectangle((j, 3 - i), 1, 1, color="#c8e6c9" if on else "white", ec="0.85"))
            ax.text(j + 0.5, 3.5 - i, "★" if on else "0", ha="center", va="center", fontsize=12)
    ax.set_xlim(0, 4); ax.set_ylim(0, 4); ax.set_aspect("equal"); ax.axis("off"); ax.set_title(title)
fig.suptitle("★ = any number (possibly non-zero); shaded = the part that may be non-zero", fontsize=9)
plt.tight_layout(); plt.show()

a11, a12, a13, a22, a23, a33 = sp.symbols("a11 a12 a13 a22 a23 a33")
T3 = sp.Matrix([[a11, a12, a13], [0, a22, a23], [0, 0, a33]])
print("det(T − λI) =", sp.factor((T3 - lam * sp.eye(3)).det()))
print("det(T)      =", T3.det())
print("eigenvalues :", list(T3.eigenvals().keys()))

🔍 **What just happened?** The three patterns show where a triangular or diagonal matrix may have non-zero entries.
For a general 3 × 3 upper triangular matrix with symbolic entries, SymPy confirms
$\det(T - \lambda I) = (a_{11}-\lambda)(a_{22}-\lambda)(a_{33}-\lambda)$, $\det T = a_{11}a_{22}a_{33}$, and the
eigenvalues $a_{11}, a_{22}, a_{33}$ (listed in reverse order) — the entries above the diagonal ($a_{12}, a_{13},
a_{23}$) do not appear at all.
They do affect the **eigenvectors**, as the next example shows.

### 6.1 Worked example: a 4 × 4 triangular matrix with a two-dimensional eigenspace

Find the eigenelements of
$$D = \begin{pmatrix}0&1&2&1\\0&1&0&2\\0&0&1&0\\0&0&0&2\end{pmatrix}.$$

*In words:* a 4 × 4 upper triangular matrix — every entry below the diagonal is 0 — with diagonal $0, 1, 1, 2$.

**Eigenvalues.** $D$ is upper triangular, so $\det(D - \lambda I_4) = (-\lambda)(1-\lambda)(1-\lambda)(2-\lambda) =
(-\lambda)(1-\lambda)^2(2-\lambda)$. (The same follows by expanding along the first column:
$-\lambda\cdot\det$ of a 3 × 3 upper triangular minor, then again.) The **distinct** eigenvalues are 0, 1 and 2, and
1 is a double root.

**$\lambda = 0$:** $D\mathbf x = \mathbf 0$ reads $x_2 + 2x_3 + x_4 = 0$, $x_2 + 2x_4 = 0$, $x_3 = 0$, $2x_4 = 0$. So
$x_4 = 0$, $x_3 = 0$, $x_2 = 0$, while $x_1$ does not appear in any equation — it is **arbitrary**. Eigenvector
$(1, 0, 0, 0)$.

**$\lambda = 1$:** $(D - I_4)\mathbf x = \mathbf 0$ reads $-x_1 + x_2 + 2x_3 + x_4 = 0$, $2x_4 = 0$, $0 = 0$, $x_4 = 0$.
So $x_4 = 0$ and $-x_1 + x_2 + 2x_3 = 0$, i.e. $x_1 = x_2 + 2x_3$ with **two** free unknowns $x_2 = s$, $x_3 = t$:

$$\mathbf x = \begin{pmatrix}s + 2t\\ s\\ t\\ 0\end{pmatrix} = s\begin{pmatrix}1\\1\\0\\0\end{pmatrix} + t\begin{pmatrix}2\\0\\1\\0\end{pmatrix}.$$

*In words:* every non-zero combination of $(1,1,0,0)$ and $(2,0,1,0)$ is an eigenvector for $\lambda = 1$; the two
vectors are a **basis of the eigenspace**, a plane in $\mathbb R^4$.

**$\lambda = 2$:** $(D - 2I_4)\mathbf x = \mathbf 0$ reads $-2x_1 + x_2 + 2x_3 + x_4 = 0$, $-x_2 + 2x_4 = 0$, $-x_3 = 0$,
$0 = 0$. So $x_3 = 0$, $x_2 = 2x_4$ and $-2x_1 + 2x_4 + x_4 = 0$, i.e. $x_1 = \tfrac32x_4$: $\mathbf x =
(\tfrac32x_4, 2x_4, 0, x_4)$; the free value $x_4 = 2$ makes the vector integer: $(3, 4, 0, 2)$.

**Multiplicities.** The **algebraic multiplicity** of an eigenvalue is how often it appears as a root of
$f(\lambda)$; its **geometric multiplicity** is the dimension of its eigenspace (the number of free variables). Always
$1\le$ geometric $\le$ algebraic. Here $\lambda = 1$ has algebraic multiplicity 2 and geometric multiplicity 2 — enough
eigenvectors. The shear ($\lambda = 1$: 2 vs 1) and the population matrix ($\lambda = -1$: 2 vs 1) fall short.

In [ ]:
# ── 6.2 The 4 × 4 example: eigenspaces by row reduction, multiplicities, and a check of every vector ──
D_sym = sp.Matrix([[0, 1, 2, 1], [0, 1, 0, 2], [0, 0, 1, 0], [0, 0, 0, 2]])
print("det(D − λI) =", sp.factor((D_sym - lam * sp.eye(4)).det()))
for value, alg_mult, vectors in D_sym.eigenvects():
    vecs = [v / sp.gcd(list(v)) for v in vectors]
    print(f"\nλ = {value}: algebraic multiplicity {alg_mult}, geometric multiplicity {len(vecs)}")
    for v in vecs:
        print(f"   eigenvector {list(v)}:  D·v = {list(D_sym * v)}")

s_, t_ = 3, -2                                             # any combination of the two λ = 1 eigenvectors ...
combo = s_ * sp.Matrix([1, 1, 0, 0]) + t_ * sp.Matrix([2, 0, 1, 0])
print(f"\n3·(1,1,0,0) − 2·(2,0,1,0) = {list(combo)};  D·combo = {list(D_sym * combo)}  (… is again an eigenvector, λ = 1)")
all_vecs = sp.Matrix.hstack(*[v for _, _, vs in D_sym.eigenvects() for v in vs])
print("rank of the 4 eigenvectors together:", all_vecs.rank())

🔍 **What just happened?** SymPy reproduces the hand calculation: $\lambda = 0$ with eigenvector $(1,0,0,0)$;
$\lambda = 1$ with algebraic **and** geometric multiplicity 2 and the basis $(1,1,0,0)$, $(2,0,1,0)$; $\lambda = 2$ with
$(3,4,0,2)$ (SymPy's own choice $(\tfrac32, 2, 0, 1)$, scaled by 2). The combination
$3(1,1,0,0) - 2(2,0,1,0) = (-1, 3, -2, 0)$ is mapped to itself — a whole plane of vectors that $D$ leaves unchanged.
Altogether the four eigenvectors have rank 4: although $D$ has only three distinct eigenvalues, it has a full basis of
eigenvectors (it is diagonalizable, [Module 13 · 🧩 Diagonalization and the singular value decomposition](13_diagonalization_svd.ipynb)).

⚠️ **Copying entries matters.** The eigen*values* of a triangular matrix do not depend on the entries above the
diagonal, but the eigen*vectors* do. If the entry $D_{34}$ were 1 instead of 0, the eigenvalues would still be
$0, 1, 1, 2$, but the eigenvector for $\lambda = 2$ would become $(5, 4, 2, 2)$. Always finish by checking
$D\mathbf v = \lambda\mathbf v$ for the matrix you were actually given.

✏️ **Your turn.** (a) Change `D_sym[2, 3]` to 1 and rerun: confirm that the eigenvalues stay the same and the
$\lambda = 2$ eigenvector becomes $(5,4,2,2)$. (b) Without computing, give the eigenvalues, determinant and trace of
$\begin{pmatrix}2&7&-1\\0&-3&5\\0&0&4\end{pmatrix}$.

### 📘 Lecture notes — triangular matrices

**Definition.** $T\in\mathbb R^{n\times n}$ is triangular if all values on one side of the diagonal are 0 (upper:
below the diagonal; lower: above it; diagonal: both).

**Properties.** $|T| = \prod_{i=1}^n T_{ii}$ (proof: expand along the first column, repeatedly) and the eigenvalues are
the diagonal entries, $\langle\lambda_1,\dots,\lambda_n\rangle = \langle T_{11},\dots,T_{nn}\rangle$, because
$T - \lambda I$ is triangular with diagonal $T_{ii} - \lambda$: for a 3 × 3 upper triangular matrix
$f(\lambda) = (a_{11}-\lambda)(a_{22}-\lambda)(a_{33}-\lambda)$, whose zeros are exactly $a_{11}, a_{22}, a_{33}$.

**Worked example.** $D = \begin{pmatrix}0&1&2&1\\0&1&0&2\\0&0&1&0\\0&0&0&2\end{pmatrix}$:
$f(\lambda) = (-\lambda)(1-\lambda)^2(2-\lambda)$, so $\lambda = 0, 1, 2$ (1 twice).
$\lambda = 0$: $(1,0,0,0)$ ($x_1$ arbitrary). $\lambda = 1$: $-x_1 + x_2 + 2x_3 = 0$, $x_4 = 0$ → basis $(1,1,0,0)$ and
$(2,0,1,0)$ of a 2-D eigenspace. $\lambda = 2$: $(\tfrac32x_4, 2x_4, 0, x_4)$, with $x_4 = 2$ chosen to make it integer:
$(3,4,0,2)$.

**Multiplicities.** Algebraic (power of the factor in $f$) ≥ geometric (dimension of the eigenspace) ≥ 1. A full set
of $n$ independent eigenvectors exists iff they are equal for every eigenvalue.

**Common confusion.** Triangular makes the eigen*values* trivial, not the eigen*vectors*: those still require solving
$(T - \lambda I)\mathbf x = \mathbf 0$, and they depend on the off-diagonal entries.

> 🎓 **Exam tip.** When asked for the "eigenvectors" of a repeated eigenvalue, give a **basis** of its eigenspace — as
> many independent vectors as there are free variables — not just one vector.

---
## 7 · Symmetric matrices, the spectral theorem and $B^\top B$

> 📍 **Where we are.** The matrices of data analysis are almost all **symmetric** ($A = A^\top$): covariance and
> correlation matrices, distance matrices, $X^\top X$, adjacency matrices of undirected networks (Modules 06 and 08).
> Symmetric matrices have the best possible eigen-behaviour, summarised by the spectral theorem. It is the theorem
> behind PCA.

**Facts for a symmetric $A\in\mathbb R^{n\times n}$.**

1. All eigenvalues are **real** (no rotations hide in a symmetric matrix).
2. Eigenvectors for different eigenvalues are **orthogonal**. Proof: if $A\mathbf x_1 = \lambda_1\mathbf x_1$ and
   $A\mathbf x_2 = \lambda_2\mathbf x_2$, then

$$\lambda_1\,\mathbf x_1^\top\mathbf x_2 = (A\mathbf x_1)^\top\mathbf x_2 = \mathbf x_1^\top A^\top\mathbf x_2
= \mathbf x_1^\top A\,\mathbf x_2 = \lambda_2\,\mathbf x_1^\top\mathbf x_2 ,$$

   *In words:* move $A$ from one side of the dot product to the other — allowed because $A^\top = A$ — and the same
   number appears multiplied once by $\lambda_1$ and once by $\lambda_2$. So $(\lambda_1-\lambda_2)\mathbf x_1^\top\mathbf x_2
   = 0$, and with $\lambda_1\neq\lambda_2$ the dot product $\mathbf x_1^\top\mathbf x_2$ must be 0.
3. **The spectral theorem.** There is an **orthonormal basis** of $\mathbb R^n$ consisting of (real) eigenvectors of
   $A$, so that

$$A = V\Lambda V^\top, \qquad V^\top V = I ,$$

   *In words:* $V$ holds the orthonormal eigenvectors as its columns (an **orthogonal** matrix: its columns have
   length 1 and are mutually perpendicular, so $V^{-1} = V^\top$, Module 11), and $\Lambda$ (capital lambda) is the
   diagonal matrix of the real eigenvalues in the same order.

**Example.** $A = \begin{pmatrix}2&1\\1&2\end{pmatrix}$: $\det\begin{pmatrix}2-\lambda&1\\1&2-\lambda\end{pmatrix} =
(2-\lambda)^2 - 1 = \lambda^2 - 4\lambda + 3 = (\lambda - 3)(\lambda - 1) = 0$, so $\lambda_1 = 3$ with
$\mathbf v_1 = (1,1)$ and $\lambda_2 = 1$ with $\mathbf v_2 = (1,-1)$, and indeed $\mathbf v_1\cdot\mathbf v_2 = 1 - 1 = 0$.

⚠️ **Common confusion — normalise before writing $V\Lambda V^\top$.** With the *unnormalised* vectors as columns,
$[\mathbf v_2, \mathbf v_1] = \begin{pmatrix}1&1\\-1&1\end{pmatrix}$, the product
$\begin{pmatrix}1&1\\-1&1\end{pmatrix}\begin{pmatrix}1&0\\0&3\end{pmatrix}\begin{pmatrix}1&-1\\1&1\end{pmatrix}$ equals
$\begin{pmatrix}4&2\\2&4\end{pmatrix} = 2A$, not $A$: each column has length $\sqrt2$, and the factor 2 sneaks in. The
spectral theorem needs **unit** eigenvectors, $\mathbf v_1 = (1,1)/\sqrt2$ and $\mathbf v_2 = (1,-1)/\sqrt2$. (With
unnormalised eigenvectors use $X\Lambda X^{-1}$ instead, [Module 13 · 🧩 Diagonalization and the singular value decomposition](13_diagonalization_svd.ipynb).)

In [ ]:
# ── 7.1 Spectral theorem for [[2,1],[1,2]]: unnormalised vs orthonormal eigenvectors ──
A_sym2 = np.array([[2.0, 1.0], [1.0, 2.0]])
Lam = np.diag([1.0, 3.0])                                     # eigenvalues in the order (λ2, λ1)
X_unnorm = np.array([[1.0, 1.0], [-1.0, 1.0]])                # columns v2 = (1, −1), v1 = (1, 1)
V = X_unnorm / np.sqrt(2)                                     # columns scaled to length 1
print("unnormalised  X Λ Xᵀ =\n", X_unnorm @ Lam @ X_unnorm.T)
print("orthonormal   V Λ Vᵀ =\n", V @ Lam @ V.T)
print("VᵀV =\n", (V.T @ V).round(12) + 0.0)
print("unnormalised  X Λ X⁻¹ =\n", X_unnorm @ Lam @ np.linalg.inv(X_unnorm))

fig, ax = plt.subplots(figsize=(4.8, 4.4))
phi = np.linspace(0, 2 * np.pi, 200)
circle = np.vstack([np.cos(phi), np.sin(phi)])
ellipse = A_sym2 @ circle
u1, u2 = np.array([1.0, 1.0]) / np.sqrt(2), np.array([-1.0, 1.0]) / np.sqrt(2)   # (sign of v2 flipped for the picture)
ax.plot(*circle, color="0.6", ls="--", label="unit circle")
ax.plot(*ellipse, color="tab:purple", label="image A·(circle)")
ax.annotate("", xy=3 * u1, xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color="tab:blue", lw=1.2, alpha=0.5))
ax.text(*(3 * u1 + [0.1, -0.35]), "A·v1 = 3 v1", color="tab:blue", fontsize=8)
plot_vectors([u1, u2], colors=["tab:blue", "tab:orange"], ax=ax, lim=(-3.4, 3.4))
ax.text(0.8, 0.35, "v1 (λ = 3)", color="tab:blue", fontsize=9)
ax.text(-1.75, 0.85, "v2 (λ = 1)", color="tab:orange", fontsize=9)
ax.plot([0.11, 0.0, -0.11], [0.11, 0.22, 0.11], color="k", lw=0.8)        # right-angle marker
ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$"); ax.legend(fontsize=7, loc="lower right")
ax.set_title("A = [[2,1],[1,2]]: orthonormal eigenvectors;\ncircle → ellipse with semi-axes 3 and 1")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The first product, with unnormalised eigenvectors, gave $\begin{pmatrix}4&2\\2&4\end{pmatrix}
= 2A$ — the trap of the warning. After dividing each column by $\sqrt2$, $V\Lambda V^\top = \begin{pmatrix}2&1\\1&2
\end{pmatrix} = A$ exactly, and $V^\top V = I$ (orthonormal columns). With unnormalised columns, $X\Lambda X^{-1}$ works
too, because $X^{-1}$ undoes whatever lengths the columns have.

The picture shows the geometry of the spectral theorem: the two unit eigenvectors are perpendicular (drawn as
$(1,1)/\sqrt2$ and $(-1,1)/\sqrt2$ — the sign of an eigenvector is free), and $A$ maps the unit circle to an
**ellipse whose axes point along the eigenvectors**, with semi-axes 3 (along $\mathbf v_1$, the faint long arrow
$A\mathbf v_1 = 3\mathbf v_1$) and 1 (along $\mathbf v_2$, which $A$ leaves unchanged). For symmetric matrices,
eigenvectors are the axes of the ellipse — the fact PCA is built on.

### 7.1 Positive semidefinite matrices and $B^\top B$

**Definition.** A symmetric matrix $M\in\mathbb R^{n\times n}$ is **positive semidefinite** (PSD, written
$M\succeq0$) if

$$\mathbf x^\top M\mathbf x \ge 0 \quad\text{for all } \mathbf x\in\mathbb R^n .$$

*In words:* the number $\mathbf x^\top M\mathbf x$ (a "quadratic form": row vector × matrix × column vector) is never
negative, whatever $\mathbf x$ you plug in. For a symmetric matrix this is equivalent to **all eigenvalues $\ge 0$**:
for an eigenvector, $\mathbf v^\top M\mathbf v = \lambda\,\mathbf v^\top\mathbf v = \lambda\|\mathbf v\|^2$, which is
$\ge0$ only if $\lambda\ge0$ (and the spectral theorem gives the converse).

**Theorem.** For **any** matrix $B\in\mathbb R^{m\times n}$ (not necessarily square), $B^\top B$ is square ($n\times n$),
symmetric and positive semidefinite.

**Proof.** Square: $(n\times m)(m\times n) = n\times n$. Symmetric: $(B^\top B)^\top = B^\top(B^\top)^\top = B^\top B$ (the
transpose of a product reverses the order, Module 08). PSD: for every $\mathbf x$,

$$\mathbf x^\top B^\top B\,\mathbf x = (B\mathbf x)^\top(B\mathbf x) = \|B\mathbf x\|^2 \ge 0 .$$

*In words:* the quadratic form of $B^\top B$ is the squared length of the vector $B\mathbf x$, and squared lengths are
never negative. ∎ This is why covariance matrices — which are $\frac1N X^\top X$ for a centred data matrix $X$ — can
never have negative eigenvalues: variances along any direction are $\ge 0$.

In [ ]:
# ── 7.2 BᵀB is symmetric PSD: a random 4 × 3 matrix and the 24 × 400 expression matrix ──
B_rand = rng.normal(size=(4, 3))
G_rand = B_rand.T @ B_rand
print("random B (4×3): BᵀB symmetric?", np.allclose(G_rand, G_rand.T),
      "| eigenvalues:", np.linalg.eigvalsh(G_rand).round(4))
quad = [x @ G_rand @ x for x in rng.normal(size=(10_000, 3))]
print(f"min of xᵀ(BᵀB)x over 10,000 random x: {min(quad):.4f}  (never negative)")

expr = simulate_tissue_expression()                    # SIMULATED infection experiment, 24 samples × 400 genes
X_expr = expr["X"]
gram = X_expr.T @ X_expr                               # 400 × 400
ev = np.linalg.eigvalsh(gram)
print(f"\nexpression XᵀX: shape {gram.shape}, symmetric? {np.allclose(gram, gram.T)}")
print(f"smallest eigenvalue {ev.min():.2e}, largest {ev.max():.3e}")
print("eigenvalues larger than 1e-6 × the largest:", int(np.sum(ev > 1e-6 * ev.max())),
      "  = rank of X =", np.linalg.matrix_rank(X_expr))

🔍 **What just happened?** For a random $4\times3$ matrix $B$, the $3\times3$ matrix $B^\top B$ is symmetric with three
positive eigenvalues, and the quadratic form $\mathbf x^\top B^\top B\mathbf x$ stayed non-negative for 10,000 random
$\mathbf x$ (a numerical test can only *illustrate* the theorem; the proof above is what guarantees it).

The simulated expression matrix ($N = 24$ samples × $J = 400$ genes) gives a $400\times400$ matrix $X^\top X$. Its
smallest eigenvalue is a tiny number of the order of the rounding error (it can even come out slightly negative in
floating point), and only **24** eigenvalues are clearly non-zero — exactly the rank of $X$. A $400\times400$ matrix
built from 24 samples cannot have rank above 24 (Module 11), so 376 eigenvalues are 0 and $X^\top X$ is singular. This
is why regression with more genes than samples breaks down ([Module 27 · 📉 Linear regression: the whole course in one model](27_linear_regression_review.ipynb), and the "$N < J$" problem of 02-620), and why
[Module 13 · 🧩 Diagonalization and the singular value decomposition](13_diagonalization_svd.ipynb) works with the singular values of $X$ instead.

### 7.2 The covariance ellipse of two interferon genes

STAT1 is a transcription factor that switches on interferon-stimulated genes such as MX1 (in the course's
inflammation network, STAT1 → MX1). Across the 24 samples of the infection experiment the two genes should therefore
rise and fall together. Their **sample covariance matrix**

$$S = \frac{1}{N-1}\sum_{i=1}^{N}(\mathbf x_i - \bar{\mathbf x})(\mathbf x_i - \bar{\mathbf x})^\top
= \begin{pmatrix}\operatorname{var}(\text{STAT1}) & \operatorname{cov}\\ \operatorname{cov} & \operatorname{var}(\text{MX1})\end{pmatrix}$$

*In words:* $\mathbf x_i$ is sample $i$'s pair (STAT1, MX1), $\bar{\mathbf x}$ the average pair; the diagonal holds the two
variances and the off-diagonal the covariance (how much they move together). It is symmetric and, being
$\frac1{N-1}X_c^\top X_c$ for the centred data $X_c$, positive semidefinite. Its eigenvectors are the axes of the data
cloud and its eigenvalues the variances along those axes.

In [ ]:
# ── 7.3 Eigenvectors of the STAT1–MX1 covariance matrix = axes of the scatter (SIMULATED data) ──
genes = expr["genes"]
pair = ["STAT1", "MX1"]
Z = X_expr[:, [genes.index(g) for g in pair]]              # 24 × 2: one row per sample
S_cov = np.cov(Z, rowvar=False)                             # 2 × 2 sample covariance (divides by N − 1)
evals, evecs = np.linalg.eigh(S_cov)                        # symmetric → eigh: ascending order
order = np.argsort(evals)[::-1]                             # largest first
evals, evecs = evals[order], evecs[:, order]
if evecs[0, 0] < 0:
    evecs[:, 0] *= -1                                       # sign of an eigenvector is arbitrary: make it point up-right
print("covariance matrix S =\n", S_cov.round(3))
print("eigenvalues (variances along the axes):", evals.round(3),
      f"→ the long axis carries {evals[0] / evals.sum():.1%} of the total variance {evals.sum():.3f} = trace S")
print("eigenvectors (columns):\n", evecs.round(3), "\northogonal? v1·v2 =", round(float(evecs[:, 0] @ evecs[:, 1]), 12))

mean = Z.mean(axis=0)
fig, ax = plt.subplots(figsize=(7.6, 4.8))
tissue_marker = {"spleen": "o", "kidney": "s", "liver": "^"}
for (tissue, cond), sub in expr["samples"].groupby(["tissue", "condition"]):
    idx = sub.index.to_numpy()
    ax.scatter(Z[idx, 0], Z[idx, 1], marker=tissue_marker[tissue], s=36,
               color="tab:red" if cond == "infected" else "tab:blue", alpha=0.8, label=f"{tissue}, {cond}")
for k, (c, offset) in enumerate([("k", (-70, 4)), ("0.35", (-40, 8))]):
    arrow = 2 * np.sqrt(evals[k]) * evecs[:, k]             # ± 2 standard deviations along each axis
    ax.annotate("", xy=mean + arrow, xytext=mean - arrow, arrowprops=dict(arrowstyle="<->", color=c, lw=2))
    ax.annotate(f"v{k + 1}: λ = {evals[k]:.2f}", mean + arrow, textcoords="offset points", xytext=offset,
                fontsize=9, color=c)
ell = mean[:, None] + evecs @ (2 * np.sqrt(evals)[:, None] * np.vstack([np.cos(phi), np.sin(phi)]))
ax.plot(*ell, color="tab:purple", lw=1, ls="--", label="2-SD ellipse")
ax.set_xlabel("STAT1 (log2 expression)"); ax.set_ylabel("MX1 (log2 expression)"); ax.set_aspect("equal")
ax.set_title("SIMULATED infection experiment: covariance eigenvectors = axes of the cloud")
ax.legend(fontsize=7, loc="upper left", bbox_to_anchor=(1.02, 1.0)); plt.tight_layout(); plt.show()

🔍 **What just happened?** The covariance matrix of STAT1 and MX1 is
$S \approx \begin{pmatrix}1.127 & 1.065\\ 1.065 & 1.699\end{pmatrix}$ (the strongly positive covariance says the genes rise
together; their correlation is $1.065/\sqrt{1.127\cdot1.699}\approx 0.77$). Its eigenvalues are $\lambda_1 \approx 2.515$
and $\lambda_2 \approx 0.310$, with orthogonal eigenvectors $\mathbf v_1 \approx (0.608, 0.794)$ and
$\mathbf v_2 \approx (-0.794, 0.608)$. In the scatter, $\mathbf v_1$ runs along the long axis of the cloud — from the
control samples (blue, bottom left) to the infected samples (red, top right): it is the shared **interferon response**
direction, along which **89 %** of the total variance ($2.515/2.825$) lies. The short axis $\mathbf v_2$ carries the
remaining variance — differences between the two genes not explained by the common response (tissue baselines,
noise). The trace check holds: $2.515 + 0.310 = 2.825 = 1.127 + 1.699$.

### 🌉 Bridge — from a 2 × 2 covariance matrix to PCA

What you just did for two genes is the first step of **principal component analysis**: centre the data, form the
covariance matrix, take its eigenvectors (the principal directions) and eigenvalues (the variance each direction
explains). With 400 genes the covariance matrix is $400\times400$, but the recipe is identical, and the spectral
theorem guarantees real variances and perpendicular axes. The 02-620 companion course derives PCA as "find the unit
vector $\mathbf u$ maximising $\mathbf u^\top S\mathbf u$", whose answer is $S\mathbf u = \lambda\mathbf u$ with the largest
$\lambda$ — see [BiologyToML · Module 03 — linear algebra for data](../../../../ml/tutorial/BiologyToML/notebooks/03_linear_algebra_for_data.ipynb) (the
covariance ellipse) and [BiologyToML · Module 14 — PCA](../../../../ml/tutorial/BiologyToML/notebooks/14_pca_dimensionality_reduction.ipynb). In [Module 13 · 🧩 Diagonalization and the singular value decomposition](13_diagonalization_svd.ipynb) you will
compute the same directions for all 400 genes at once with the singular value decomposition and watch them separate
tissues and infection status.

✏️ **Your turn.** Repeat the analysis for the pair (TNF, IL1B) — two cytokines of the NF-κB loop — and for (IL6, SAA1).
Which pair has the most elongated cloud (largest share $\lambda_1/(\lambda_1+\lambda_2)$), and how does that relate to
their correlation?

### 📘 Lecture notes — symmetric matrices, the spectral theorem and positive semidefiniteness

**Symmetric matrices** ($A = A^\top$): all eigenvalues are real, and eigenvectors of different eigenvalues are orthogonal
($\lambda_1\mathbf x_1^\top\mathbf x_2 = \mathbf x_1^\top A\mathbf x_2 = \lambda_2\mathbf x_1^\top\mathbf x_2$); they can be
chosen orthonormal.

**Spectral theorem.** If $A\in\mathbb R^{n\times n}$ is symmetric, there is an orthonormal basis of $\mathbb R^n$ consisting
of real eigenvectors of $A$, and

$$A = V\Lambda V^\top ,$$

*In words:* $V$ is an orthonormal (orthogonal) matrix whose columns are eigenvectors, $V^\top V = I$, and $\Lambda$ is a
real diagonal matrix of the eigenvalues.

**Worked example.** $A = \begin{pmatrix}2&1\\1&2\end{pmatrix}$: $(2-\lambda)^2 - 1 = \lambda^2 - 4\lambda + 3 =
(\lambda-3)(\lambda-1)$; $\lambda_1 = 3$, $\mathbf v_1 = (1,1)$; $\lambda_2 = 1$, $\mathbf v_2 = (1,-1)$;
$A = [\mathbf v_2,\mathbf v_1]\Lambda[\mathbf v_2,\mathbf v_1]^\top$ with $\Lambda = \mathrm{diag}(1, 3)$ **after normalising**
the columns: $\frac{1}{\sqrt2}\begin{pmatrix}1&1\\-1&1\end{pmatrix}\begin{pmatrix}1&0\\0&3\end{pmatrix}\frac{1}{\sqrt2}
\begin{pmatrix}1&-1\\1&1\end{pmatrix} = A$ (without the factors $\frac1{\sqrt2}$ the product is $2A$).

**Positive semidefinite.** $M\succeq 0 \iff \mathbf x^\top M\mathbf x \ge 0\ \forall\,\mathbf x\in\mathbb R^n$ $\iff$ (for
symmetric $M$) all eigenvalues $\ge 0$. **For any $B\in\mathbb R^{m\times n}$, $B^\top B$ is square, symmetric and PSD**,
because $\mathbf x^\top B^\top B\mathbf x = \|B\mathbf x\|^2\ge0$. Covariance matrices are PSD; their eigenvectors are the
axes of the data cloud and their eigenvalues the variances along those axes.

**Numerics.** `np.linalg.eigh` is for symmetric matrices: real output, orthonormal eigenvectors, eigenvalues in
**ascending** order. `np.linalg.eig` handles any square matrix, may return complex numbers, and uses no particular
order. Both scale eigenvectors to length 1, with an arbitrary sign.

**Common confusions.** $V\Lambda V^\top$ needs orthonormal columns (otherwise use $X\Lambda X^{-1}$); "PSD" refers to the
sign of $\mathbf x^\top M\mathbf x$, not to the entries of $M$ ($\begin{pmatrix}1&-1\\-1&1\end{pmatrix}$ is PSD with negative
entries).

> 🎓 **Exam tip.** To show a matrix is PSD, write it as $B^\top B$ or show all eigenvalues of the symmetric matrix are
> $\ge 0$. To find the "first principal axis", take the eigenvector of the **largest** eigenvalue — the *last* column of
> `eigh`.

---
## 8 · Computing eigenvectors in practice: `eig`, `eigh` and power iteration

> 📍 **Where we are.** By hand we used the characteristic polynomial. Computers do not: for $n\ge5$ there is no
> formula for the roots of a polynomial (Abel–Ruffini theorem), and tiny rounding errors in the coefficients can move
> the roots a lot. Numerical libraries use iterative algorithms instead. The simplest, **power iteration**, is worth
> implementing yourself: it explains why populations settle into stable age structures, how Google ranked web pages,
> and how Markov chains forget their starting state.

In [ ]:
# ── 8.1 np.linalg.eig vs np.linalg.eigh: order, normalisation, sign ──
vals_eig, vecs_eig = np.linalg.eig(A_sym2)
vals_eigh, vecs_eigh = np.linalg.eigh(A_sym2)
print("eig : eigenvalues", vals_eig, "\n      eigenvectors (columns)\n", vecs_eig.round(4))
print("eigh: eigenvalues", vals_eigh, "(ascending!)\n      eigenvectors (columns)\n", vecs_eigh.round(4))
print("column lengths:", np.linalg.norm(vecs_eigh, axis=0))
print("eig of a rotation returns complex numbers:", np.linalg.eig(MAPS["A3 (rotation 30°)"])[0].round(3))

🔍 **What just happened?** For the symmetric $A = \begin{pmatrix}2&1\\1&2\end{pmatrix}$ both functions find 3 and 1,
but `eig` lists them in whatever order its algorithm produced (here 3 first) while `eigh` sorts them **ascending**
(1, then 3). The eigenvectors are scaled to length 1 ($\pm(0.7071, 0.7071)$ is $(1,1)/\sqrt2$), and their signs are
arbitrary — `eigh`'s first column is $(-0.7071, 0.7071)$, the same eigen-line as $(1,-1)$. Rules of thumb: use `eigh`
for symmetric matrices (faster, real output, orthonormal vectors), reverse its order when you want the largest first,
and never compare eigenvectors without fixing their sign or scale.

### 8.1 Power iteration

**Algorithm.** Start from any vector $\mathbf x_0$ and repeat

$$\mathbf x_{k+1} = \frac{A\mathbf x_k}{\|A\mathbf x_k\|} .$$

*In words:* multiply by the matrix and rescale to length 1, over and over. The direction converges to the eigenvector
of the eigenvalue with the **largest absolute value** (the *dominant* eigenvalue), and the Rayleigh quotient
$\mathbf x_k^\top A\mathbf x_k$ converges to that eigenvalue.

**Why it works.** Suppose $A$ has a basis of eigenvectors $\mathbf v_1,\dots,\mathbf v_n$ with
$|\lambda_1| > |\lambda_2| \ge \dots$. Write the start in that basis, $\mathbf x_0 = c_1\mathbf v_1 + \dots + c_n\mathbf v_n$.
Using $A^k\mathbf v_i = \lambda_i^k\mathbf v_i$ (Section 5.4):

$$A^k\mathbf x_0 = c_1\lambda_1^k\mathbf v_1 + c_2\lambda_2^k\mathbf v_2 + \dots
= \lambda_1^k\left(c_1\mathbf v_1 + c_2\left(\tfrac{\lambda_2}{\lambda_1}\right)^{k}\mathbf v_2 + \dots\right).$$

*In words:* every component except the first is multiplied by a power of a ratio smaller than 1 in size, so it fades
away; the error shrinks by the factor $|\lambda_2/\lambda_1|$ per step. It fails if $c_1 = 0$ (a start with no
$\mathbf v_1$ component — rare) or if two eigenvalues tie in absolute value.

**Worked example.** $A = \begin{pmatrix}1&2\\3&2\end{pmatrix}$ has eigenelements $\mathbf v_1 = (1, \tfrac32)$ with
$\lambda_1 = 4$ and $\mathbf v_2 = (4,-4)$ with $\lambda_2 = -1$. Look at what $A$ does to $\mathbf u = (1,2)$:
$A\mathbf u = (5,7)$ and $A(A\mathbf u) = (19, 29)$. The ratio of the components, $7/5 = 1.4$, then $29/19 = 1.526$, is
already approaching $\tfrac32$ — the direction of $\mathbf v_1$.

In [ ]:
# ── 8.2 Power iteration, implemented and tested on A = [[1,2],[3,2]] ──
def power_iteration(M, x0, n_iter=30):
    """Repeat x ← Mx/‖Mx‖. Returns the final unit vector, the Rayleigh-quotient estimate of the dominant
    eigenvalue, and the list of all iterates (for plotting the convergence)."""
    x = np.asarray(x0, dtype=float)
    x = x / np.linalg.norm(x)
    history = [x]
    for _ in range(n_iter):
        y = M @ x
        x = y / np.linalg.norm(y)
        history.append(x)
    return x, float(x @ M @ x), history


A_pow = np.array([[1.0, 2.0], [3.0, 2.0]])
u = np.array([1.0, 2.0])
print("unnormalised iterates A^k u and the ratio of their components (→ 1.5):")
y = u.copy()
for k in range(1, 7):
    y = A_pow @ y
    print(f"  k = {k}:  A^k u = {y.astype(int)},  ratio = {y[1] / y[0]:.6f}")

x_final, lam_est, hist = power_iteration(A_pow, u, n_iter=20)
v1 = np.array([1.0, 1.5]) / np.linalg.norm([1.0, 1.5])
print(f"\npower iteration after 20 steps: x = {x_final.round(6)},  Rayleigh quotient = {lam_est:.6f}")
print(f"true unit eigenvector v1 = {v1.round(6)},  λ1 = 4")

err = [np.linalg.norm(h - v1) for h in hist]
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.semilogy(err, "o-", ms=4, label="distance of x_k from v1")
ax.semilogy(err[0] * 0.25 ** np.arange(len(err)), "--", color="0.5", label="(|λ2/λ1|)^k = (1/4)^k")
ax.set_xlabel("iteration k"); ax.set_ylabel("error (log scale)")
ax.set_title("power iteration: the error shrinks by |λ2/λ1| = 1/4 per step")
ax.legend(); plt.tight_layout(); plt.show()

🔍 **What just happened?** The unnormalised iterates $(5,7)$, $(19,29)$, $(77,115)$, … reproduce the worked example,
and the ratio of their components — 1.4, 1.526, 1.494, 1.502, … — closes in on $\tfrac32$ from alternating sides. The
alternation is the $\lambda_2 = -1$ component flipping sign every step while shrinking relative to the $4^k$ growth of
the $\mathbf v_1$ component. After 20 normalised steps, power iteration returns $(0.5547, 0.8321) = (1, 1.5)/\|(1,1.5)\|$
and the Rayleigh quotient 4.000000. On the log scale the error falls along a straight line parallel to $(1/4)^k$:
**every step gains a factor 4** — the ratio $|\lambda_1/\lambda_2|$. Matrices whose two largest eigenvalues are close
converge slowly. (In [Module 13 · 🧩 Diagonalization and the singular value decomposition](13_diagonalization_svd.ipynb) you will see the same computation in the eigenbasis: $\mathbf u = 1.2\,\mathbf v_1 - 0.05\,\mathbf v_2$.)

### 8.2 Hub genes: eigenvector centrality of the inflammation network

In a network, a gene is important if it is connected to other important genes. Writing $c_i$ for the importance of
gene $i$ and $A$ for the (undirected) adjacency matrix, "proportional to the sum of the neighbours' importances" reads

$$\lambda\,c_i = \sum_{j} A_{ij}\,c_j \qquad\Longleftrightarrow\qquad A\mathbf c = \lambda\mathbf c .$$

*In words:* the importance scores form an eigenvector of the adjacency matrix; we take the one for the largest
eigenvalue, whose entries can all be chosen non-negative. This is **eigenvector centrality**; Google's PageRank is a
variant for directed links. We compute it by power iteration on the inflammation network of Module 06, treating each
regulatory link as an undirected connection.

In [ ]:
# ── 8.3 Eigenvector centrality: power iteration on A fails, on A + I it works ──
nodes = []
for u_, v_ in INFLAMMATION_NETWORK_EDGES:
    for g_ in (u_, v_):
        if g_ not in nodes:
            nodes.append(g_)
undirected = sorted({tuple(sorted(e)) for e in INFLAMMATION_NETWORK_EDGES})   # NFKB1⇄TNF counts once
A_net = adjacency_matrix(nodes, undirected).astype(float)
spectrum = np.linalg.eigvalsh(A_net)
print(f"{len(nodes)} genes, {len(undirected)} undirected links; largest and smallest eigenvalue: "
      f"{spectrum.max():.4f}, {spectrum.min():.4f}")

x0 = np.ones(len(nodes))
x_a, q_a, hist_a = power_iteration(A_net, x0, n_iter=200)
x_b, q_b, hist_b = power_iteration(A_net + np.eye(len(nodes)), x0, n_iter=200)
print(f"power iteration on A    : Rayleigh quotient {q_a:.4f}; last two iterates differ by "
      f"{np.linalg.norm(hist_a[-1] - hist_a[-2]):.3f}")
print(f"power iteration on A + I: Rayleigh quotient {q_b:.4f} → λ_max(A) = {q_b - 1:.4f}; last two differ by "
      f"{np.linalg.norm(hist_b[-1] - hist_b[-2]):.1e}")

w_ref, V_ref = np.linalg.eigh(A_net)
centrality = pd.Series(np.abs(x_b), index=nodes).sort_values(ascending=False)
print("\neigenvector centrality (power iteration on A + I) vs eigh:")
print(pd.DataFrame({"power iteration": centrality,
                    "eigh": pd.Series(np.abs(V_ref[:, -1]), index=nodes)[centrality.index]}).round(3).T.to_string())

layout = {"NFKB1": (0, 0), "IL1B": (-0.9, 0.9), "TNF": (-1.3, 0), "CXCL8": (-0.9, -0.9), "LYZ": (0, 1.1),
          "IL6": (1.2, 0), "STAT3": (2.4, 0), "SAA1": (3.3, 0.9), "HAMP": (3.6, 0), "C3": (3.3, -0.9),
          "STAT1": (1.2, -1.7), "IRF1": (2.2, -1.3), "ISG15": (2.2, -2.2), "MX1": (0.2, -1.9)}
fig, ax = plt.subplots(figsize=(7.0, 4.4))
cmap = plt.cm.YlOrRd
draw_graph(undirected, nodes=nodes, pos=layout, ax=ax, font_size=7,
           node_colors={g_: cmap(0.12 + 0.60 * centrality[g_] / centrality.max()) for g_ in nodes},
           title="inflammation network (illustrative): colour = eigenvector centrality")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The undirected network has 14 genes and 13 links; its largest eigenvalue is 2.3702 and its
smallest is $-2.3702$ — **a tie in absolute value**. Power iteration on $A$ therefore never settles: the iterates keep
jumping between two vectors (the last two differ by 0.61), and the Rayleigh quotient (1.93) is not an eigenvalue at
all. The reason is structural: the NF-κB part of the network is a tree, and trees (more generally, bipartite graphs,
Module 06) have spectra symmetric about 0.

The fix uses Section 5.4: $A + I$ has the **same eigenvectors** with eigenvalues shifted by $+1$, i.e. 3.37 and
$-1.37$ — no more tie. Power iteration on $A + I$ converges (after 200 steps consecutive iterates agree to about
$2\times10^{-7}$; convergence is slowish because the interferon module has an eigenvalue 2.17, close to 2.37), and
subtracting the shift gives $\lambda_{\max}(A) = 2.3702$. The centrality scores agree with `eigh` to three decimals:
**NFKB1** (0.602) is the hub, followed by IL6 (0.411, the link from the NF-κB module to the acute-phase module) and
STAT3 (0.372); the cytokines and LYZ that hang off NFKB1 share 0.254. The interferon module (STAT1, IRF1, MX1, ISG15)
scores **0**: it is not connected to the NF-κB component, and the leading eigenvector lives entirely on the component
with the largest eigenvalue — a known limitation of eigenvector centrality on disconnected networks.

### 🌉 Bridge — hub genes, PageRank and graph neural networks

Eigenvector centrality and its relatives rank proteins in interaction networks, genes in co-expression networks and
cells in neighbourhood graphs; PageRank applies power iteration to a "random surfer" matrix and needs a damping term
precisely to avoid the ties and dead ends you just met. Graph neural networks repeatedly multiply node features by a
(normalised) adjacency matrix — one step of power iteration per layer — which is why too many layers make all nodes
look alike ("over-smoothing"): see [BiologyToML · Module 28 — graph neural networks](../../../../ml/tutorial/BiologyToML/notebooks/28_graph_neural_networks.ipynb)
and, for the graphs themselves, [Module 06 · 🕸️ Graphs and trees: networks and phylogenies](06_graphs_and_trees.ipynb).

### 8.3 The base composition a DNA Markov chain settles into

A first-order Markov chain for DNA (Module 20) has a $4\times4$ transition matrix $P$ with $P_{ab} = $ probability that
base $b$ follows base $a$; every row sums to 1. If $\boldsymbol\pi_t$ is the probability distribution of the base at
position $t$ (a **row** vector), the next position has $\boldsymbol\pi_{t+1} = \boldsymbol\pi_t P$. A **stationary
distribution** does not change any more:

$$\boldsymbol\pi P = \boldsymbol\pi \qquad\Longleftrightarrow\qquad P^\top\boldsymbol\pi^\top = 1\cdot\boldsymbol\pi^\top .$$

*In words:* transposing both sides turns the row-vector equation into an ordinary eigenvector equation:
$\boldsymbol\pi^\top$ is an eigenvector of $P^\top$ with **eigenvalue 1** (a "left eigenvector" of $P$), rescaled so that
its entries sum to 1. Eigenvalue 1 always exists: $P\mathbf 1 = \mathbf 1$ because every row sums to 1, and $P^\top$ has
the same eigenvalues as $P$ (Section 5's exercise).

In [ ]:
# ── 8.4 Stationary base composition = eigenvector of Pᵀ with eigenvalue 1 (and a SIMULATED check) ──
def stationary_distribution(P):
    """Left eigenvector of a transition matrix P for eigenvalue 1, rescaled to sum to 1."""
    vals, vecs = np.linalg.eig(np.asarray(P).T)
    k = np.argmin(np.abs(vals - 1.0))
    pi = np.real(vecs[:, k])
    return pi / pi.sum(), vals


rows = {}
for name, P in [("background", BACKGROUND_TRANSITIONS), ("CpG island", CPG_ISLAND_TRANSITIONS)]:
    pi, vals = stationary_distribution(P)
    seq = simulate_dna(20_000, transitions=P, seed=7)          # SIMULATED 20 kb sequence from this chain
    observed = np.array([seq.count(b) for b in "ACGT"]) / len(seq)
    rows[name + ": stationary π"] = pi
    rows[name + ": simulated 20 kb"] = observed
    print(f"{name:11s} eigenvalues of Pᵀ: {np.round(vals, 3)};  π·P − π = {np.abs(pi @ P - pi).max():.1e}")
print(pd.DataFrame(rows, index=list("ACGT")).T.round(4).to_string())

🔍 **What just happened?** Both transition matrices have eigenvalue 1 (the other eigenvalues are smaller in absolute
value, some complex). The background chain settles into an almost uniform base composition (A 0.263, C 0.247, G 0.238,
T 0.252), while the CpG-island chain settles into a **GC-rich** composition (C 0.343, G 0.347; together 69 %) —
consistent with the high C+G content that defines CpG islands. The base frequencies of 20,000 simulated bases match
the stationary distributions to about two decimals, as they should for a long sequence: the chain "forgets" its first
base, just as the population of Section 0 forgot its starting counts — both are power iterations in disguise
($\boldsymbol\pi_t = \boldsymbol\pi_0P^t$).

### 🌉 Bridge — stationary distributions and Markov chains

[Module 20 · ⛓️ Graphical models: Bayesian networks and Markov chains](20_bayesian_networks_markov_chains.ipynb) builds Markov chains from scratch (the CpG-island model, the weather and telephone examples) and computes the
long-run distribution both as the limit of $P^n$ and as this eigenvector. Hidden Markov models for gene finding and
sequence evolution models rest on the same matrices — see
[BiologyToML · Module 19 — HMM inference](../../../../ml/tutorial/BiologyToML/notebooks/19_hmm_inference.ipynb).

✏️ **Your turn.** Run `power_iteration(BACKGROUND_TRANSITIONS.T, np.ones(4))` and rescale the result to sum to 1. Do you
recover π? How many iterations are needed, and which eigenvalue ratio controls that number?

---
## 9 · Population growth: the Leslie matrix

> 📍 **Where we are.** Everything comes together: the characteristic polynomial (Section 3), powers $A^k$ (Section 5)
> and power iteration (Section 8) explain the population projection we started with.

**The model.** A **Leslie matrix** for $m$ age classes has the fecundities $F_0, F_1, \dots, F_{m-1}$ (offspring per
individual per year) in its first row and the survival probabilities $S_0, \dots, S_{m-2}$ just below the diagonal:

$$L = \begin{pmatrix}F_0 & F_1 & \cdots & F_{m-2} & F_{m-1}\\ S_0 & 0 & \cdots & 0 & 0\\ 0 & S_1 & \cdots & 0 & 0\\
\vdots & & \ddots & & \vdots\\ 0 & 0 & \cdots & S_{m-2} & 0\end{pmatrix},\qquad \mathbf n_{t+1} = L\,\mathbf n_t,\qquad
\mathbf n_t = L^t\,\mathbf n_0 .$$

*In words:* next year's newborns are the fecundity-weighted sum of all classes; everybody else is a survivor of the
class below. Projecting $t$ years ahead is multiplying by $L$ $t$ times — power iteration without the rescaling.

**The long run.** If $\mathbf n_0 = c_1\mathbf v_1 + c_2\mathbf v_2 + \dots$ in eigenvectors (Section 8.1), then
$\mathbf n_t = c_1\lambda_1^t\mathbf v_1 + c_2\lambda_2^t\mathbf v_2 + \dots \approx c_1\lambda_1^t\mathbf v_1$ for large $t$,
provided $|\lambda_1|$ is strictly the largest. So:

* the **dominant eigenvalue** $\lambda_1$ is the long-run **growth factor per year** (the population grows if
  $\lambda_1 > 1$, is stable if $\lambda_1 = 1$, declines if $\lambda_1 < 1$);
* the **dominant eigenvector** $\mathbf v_1$, rescaled to sum to 1, is the **stable age distribution**.

For a Leslie matrix with non-negative entries the Perron–Frobenius theorem guarantees (under mild conditions — every
class can contribute to every other, and the reproducing ages have no common period) that $\lambda_1$ is real,
positive and strictly largest in size, with an eigenvector of positive entries — a biologically meaningful population.
Our matrix: $\lambda_1 = 2$ with $\mathbf v_1 = (16,4,1)$; the other eigenvalue, $-1$ (twice), has $|-1| < 2$.

In [ ]:
# ── 9.1 The projection of Section 0 explained: growth factor → λ1, proportions → v1/Σv1 ──
vals_L, vecs_L = np.linalg.eig(L)
k1 = np.argmax(np.abs(vals_L))
lam1 = vals_L[k1].real
v1_L = np.abs(vecs_L[:, k1].real)
stable = v1_L / v1_L.sum()
print(f"eigenvalues of L: {np.round(vals_L.real, 6)};  dominant λ1 = {lam1:.6f}")
print(f"stable age distribution v1/Σv1 = {stable.round(4)}  (= (16, 4, 1)/21 = {np.round(np.array([16, 4, 1]) / 21, 4)})")
print("year 14 proportions           =", (pop.loc[14, ["juveniles", "age1", "age2"]] / pop.loc[14, "total"]).to_numpy().round(4))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
ax = axes[0]
for col, c in zip(["juveniles", "age1", "age2"], ["tab:blue", "tab:orange", "tab:green"]):
    counts = pop[col].where(pop[col] > 0)                      # zero counts cannot be drawn on a log axis → gaps
    ax.semilogy(pop.index, counts, "o-", ms=3, color=c, label=col)
ax.semilogy(pop.index, 10 * 2.0 ** pop.index * 0.6, "k--", lw=1, label="slope of 2^t")
ax.set_xlabel("year t"); ax.set_ylabel("number of animals (log scale)"); ax.legend(fontsize=7)
ax.set_title("counts per age class (illustrative)")
ax = axes[1]
ax.plot(pop.index[1:], pop["growth"].iloc[1:], "o-", ms=4, label="N(t)/N(t−1)")
ax.axhline(lam1, color="tab:red", ls="--", label=f"λ1 = {lam1:.0f}")
ax.set_xlabel("year t"); ax.set_ylabel("yearly growth factor"); ax.set_ylim(0, 8.5); ax.legend()
ax.set_title("the growth factor converges to the dominant eigenvalue")
plt.tight_layout(); plt.show()

🔍 **What just happened?** NumPy confirms the eigenvalues $2, -1, -1$ and the stable age distribution
$(0.7619, 0.1905, 0.0476) = (16,4,1)/21$; the year-14 proportions of the projection, $(0.7615, 0.1911, 0.0474)$,
agree to about three decimals. On the log scale (gaps = years in which a class is empty) all three age classes end up
parallel to the dashed line of slope $2^t$, and the yearly growth factor zig-zags into $\lambda_1 = 2$. The zig-zag is the eigenvalue $-1$: its component flips sign
every year, while its relative weight shrinks like $t\,(1/2)^t$ (the factor $t$ comes from the missing second
eigenvector of the double root — [Module 13 · 🧩 Diagonalization and the singular value decomposition](13_diagonalization_svd.ipynb)).

### 9.1 A second, more realistic Leslie matrix and a conservation question

Consider an (illustrative) longer-lived species with four age classes: first-years do not breed; ages 1, 2 and 3 produce
1.0, 2.0 and 1.5 surviving offspring per year; the survival probabilities from one class to the next are 0.5, 0.7 and
0.5. Is the population viable — and what happens if a new predator halves first-year survival? Can protecting nests
(raising fecundity) compensate?

In [ ]:
# ── 9.2 Growth rate and stable age structure of a 4-class Leslie matrix; a conservation scenario (illustrative) ──
def leslie(fecundity, survival):
    """Leslie matrix from fecundities (first row) and survival probabilities (sub-diagonal)."""
    m = len(fecundity)
    M = np.zeros((m, m))
    M[0] = fecundity
    M[np.arange(1, m), np.arange(m - 1)] = survival
    return M


def dominant(M):
    """Dominant eigenvalue and the corresponding eigenvector rescaled to sum to 1."""
    vals, vecs = np.linalg.eig(M)
    k = np.argmax(np.abs(vals))
    v = np.abs(vecs[:, k].real)
    return vals[k].real, v / v.sum()


F_base, S_base = np.array([0, 1.0, 2.0, 1.5]), np.array([0.5, 0.7, 0.5])
S_pred = S_base * [0.5, 1, 1]                                  # a predator halves first-year survival
scenarios = {"baseline": leslie(F_base, S_base),
             "predator halves first-year survival": leslie(F_base, S_pred),
             "predator + nest protection (fecundity ×1.5)": leslie(1.5 * F_base, S_pred),
             "predator + nest protection (fecundity ×2)": leslie(2.0 * F_base, S_pred)}
fig, ax = plt.subplots(figsize=(6.4, 3.6))
for name, M in scenarios.items():
    lam1_s, stable_s = dominant(M)
    print(f"{name:44s} λ1 = {lam1_s:.4f} ({100 * (lam1_s - 1):+.1f} % per year);  stable ages = {stable_s.round(3)}")
    n_t = np.array([100.0, 50.0, 25.0, 10.0])
    totals = [n_t.sum()]
    for _ in range(30):
        n_t = M @ n_t
        totals.append(n_t.sum())
    ax.semilogy(totals, label=f"{name} (λ1 = {lam1_s:.3f})")
ax.set_xlabel("year"); ax.set_ylabel("total population (log scale)")
ax.set_title("illustrative 4-age-class population under four scenarios")
ax.legend(fontsize=7); plt.tight_layout(); plt.show()

🔍 **What just happened?** The baseline population has $\lambda_1 = 1.1452$: it grows by 14.5 % per year, and in the
long run 55 % of the animals are first-years. Halving first-year survival pushes $\lambda_1$ down to 0.8965 — a
**decline** of 10.4 % per year, i.e. extinction in the long run. Nest protection that raises fecundity by 50 % brings
$\lambda_1$ back above 1 (1.0332, +3.3 % per year); doubling fecundity restores **exactly** the baseline growth rate
1.1452 — although the age structure is different (more first-years, 71 %). That is no coincidence: expanding
$\det(L - \lambda I)$ for a Leslie matrix with $F_0 = 0$ gives the characteristic equation

$$\lambda^4 = F_1S_0\,\lambda^2 + F_2S_0S_1\,\lambda + F_3S_0S_1S_2 ,$$

*In words:* every route to a new juvenile passes through first-year survival $S_0$ and one fecundity, so $\lambda_1$
depends on $S_0$ and the $F_i$ only through the products $S_0F_i$ — halving one and doubling the other cancel.
On the log scale every trajectory becomes a straight line with slope $\log\lambda_1$ after a short transient. One
eigenvalue summarises the fate of the population — which is why conservation biologists compute exactly this number
(and its *sensitivity* to each vital rate) to decide which life stage to protect.

### 🌉 Bridge — stability of equilibria

The population model is a linear **discrete-time dynamical system** $\mathbf n_{t+1} = L\mathbf n_t$. Its equilibrium
$\mathbf n = \mathbf 0$ (extinction) is **stable** — every population shrinks back to it — exactly when **all eigenvalues
satisfy $|\lambda| < 1$**, and unstable as soon as one eigenvalue has $|\lambda| > 1$. For continuous-time models
$\dot{\mathbf x} = M\mathbf x$ (ordinary differential equations), the rule becomes "all eigenvalues have negative real part",
because the component along an eigenvector evolves as $e^{\lambda t}$. [Module 15 · ⛰️ Gradients, Jacobians, optimization and dynamics](15_multivariable_calculus_dynamics.ipynb) uses this to decide whether a tRNA-charging
model or an epidemic (SIR) settles or explodes after a perturbation, and the 02-620 companion uses it for cell-state
dynamics — [BiologyToML · Module 32 — cell dynamics and RNA velocity](../../../../ml/tutorial/BiologyToML/notebooks/32_cell_dynamics_velocity.ipynb). In
[Module 13 · 🧩 Diagonalization and the singular value decomposition](13_diagonalization_svd.ipynb) you will compute $L^t$ in closed form by diagonalization.

✏️ **Your turn.** For the three-class insect, replace both survival probabilities $\tfrac12$ by a common value $s$.
Show that the characteristic equation becomes $\lambda^3 - 6s\lambda - 8s^2 = 0$, and find the survival $s$ for which
$\lambda_1 = 1$ (the population exactly replaces itself). Check your answer numerically with `dominant`.

---
## 📌 Key takeaways

- **Definition.** $A\mathbf x = \lambda\mathbf x$ with $\mathbf x\neq\mathbf 0$: the matrix only stretches, shrinks or
  flips an eigenvector; $A\mathbf x$ stays on the line through $\mathbf 0$ and $\mathbf x$. Verify by one multiplication.
- **Finding them.** $(A-\lambda I)\mathbf x = \mathbf 0$ has non-zero solutions iff $A - \lambda I$ is **singular** iff
  $\det(A-\lambda I) = 0$ — the characteristic equation. Roots = eigenvalues; null spaces = eigenvectors.
  2 × 2: $\lambda^2 - \operatorname{tr}A\,\lambda + \det A = 0$.
- **Worked examples.** $\begin{pmatrix}3&4\\2&1\end{pmatrix}$: $\lambda = 5, -1$ with $(2,1), (1,-1)$.
  $B = \begin{pmatrix}1&1&-1\\2&3&-4\\4&1&-4\end{pmatrix}$: $\lambda^3 - 7\lambda + 6 = 0$, $\lambda = 1, 2, -3$ with
  $(1,1,1), (1,2,1), (1,7,11)$.
- **Pictures.** Real distinct eigenvalues: two invariant lines; rotation: complex eigenvalues, no real eigenvector;
  shear: a repeated eigenvalue with one direction; a zero eigenvalue collapses a dimension.
- **Properties.** Distinct eigenvalues ⇒ independent eigenvectors; $\det A = \prod\lambda_i$, $\operatorname{tr}A =
  \sum\lambda_i$; $A$ invertible iff $0$ is not an eigenvalue; $A^{-1}, A^k, A + cI$ have eigenvalues
  $1/\lambda, \lambda^k, \lambda + c$ with the same eigenvectors.
- **Triangular matrices** have their eigenvalues on the diagonal; a repeated eigenvalue may have a multi-dimensional
  eigenspace (give a basis).
- **Symmetric matrices** have real eigenvalues and orthonormal eigenvectors, $A = V\Lambda V^\top$ (normalise!).
  $B^\top B$ is always symmetric PSD; covariance eigenvectors are the axes of the data cloud (PCA).
- **Numerics.** `eigh` for symmetric matrices (ascending order), `eig` otherwise; power iteration finds the dominant
  eigenvector at rate $|\lambda_2/\lambda_1|$ and fails on ties — shift by $cI$.
- **Biology.** Dominant eigenvalue = long-run growth factor, dominant eigenvector = stable age structure; eigenvector
  centrality finds hub genes; the eigenvector with eigenvalue 1 of $P^\top$ is the stationary base composition.

---
## ✅ Checkpoint

1. Is $(1,-1)$ an eigenvector of $\begin{pmatrix}3&4\\2&1\end{pmatrix}$? Of $\begin{pmatrix}2&2\\-4&8\end{pmatrix}$?
   <details><summary>Answer</summary>First matrix: $(3-4,\ 2-1) = (-1, 1) = -1\cdot(1,-1)$ — yes, with $\lambda = -1$. Second
   matrix: $(2-2,\ -4-8) = (0,-12)$, not a multiple of $(1,-1)$ — no.</details>
2. Find all eigenvalues and eigenvectors of $\begin{pmatrix}4&1\\2&3\end{pmatrix}$ by hand, and of
   $\begin{pmatrix}2&2\\-4&8\end{pmatrix}$.
   <details><summary>Answer</summary>$\lambda^2 - 7\lambda + 10 = (\lambda-2)(\lambda-5)$. $\lambda = 5$:
   $\begin{pmatrix}-1&1\\2&-2\end{pmatrix}\mathbf x = \mathbf 0 \Rightarrow (1,1)$; $\lambda = 2$:
   $\begin{pmatrix}2&1\\2&1\end{pmatrix}\mathbf x = \mathbf 0 \Rightarrow (1,-2)$. Second matrix: $\lambda^2 - 10\lambda + 24 =
   (\lambda-4)(\lambda-6)$; $\lambda=4$: $(1,1)$; $\lambda = 6$: $\begin{pmatrix}-4&2\\-4&2\end{pmatrix}\mathbf x = \mathbf 0
   \Rightarrow (1,2)$.</details>
3. A classmate writes: "$(A-\lambda I)\mathbf x = \mathbf 0$ and $\mathbf x\neq\mathbf 0$, so $A - \lambda I = 0$." What is
   wrong, and what is the correct conclusion?
   <details><summary>Answer</summary>A non-zero matrix can map a non-zero vector to $\mathbf 0$ (e.g. $\begin{pmatrix}1&1\\1&1
   \end{pmatrix}(1,-1)^\top = \mathbf 0$). The correct conclusion is that $A-\lambda I$ is **singular**: if it were invertible,
   $\mathbf x = (A-\lambda I)^{-1}\mathbf 0 = \mathbf 0$. Singular ⇔ $\det(A - \lambda I) = 0$.</details>
4. Expand $\det(B - \lambda I)$ for $B = \begin{pmatrix}1&1&-1\\2&3&-4\\4&1&-4\end{pmatrix}$ along row 1 and confirm the
   characteristic polynomial.
   <details><summary>Answer</summary>$(1-\lambda)\big[(3-\lambda)(-4-\lambda) + 4\big] - 1\big[2(-4-\lambda) + 16\big] + (-1)\big[2 - 4(3-\lambda)\big]
   = (1-\lambda)(\lambda^2 + \lambda - 8) - (8 - 2\lambda) - (4\lambda - 10) = -\lambda^3 + 9\lambda - 8 - 8 + 2\lambda - 4\lambda + 10
   = -\lambda^3 + 7\lambda - 6$. ✓</details>
5. Without any computation, give the eigenvalues, determinant and trace of $\begin{pmatrix}2&7&-1\\0&-3&5\\0&0&4\end{pmatrix}$.
   Is it invertible?
   <details><summary>Answer</summary>Triangular: $\lambda = 2, -3, 4$; $\det = 2\cdot(-3)\cdot4 = -24$; trace $= 3$. Invertible,
   because 0 is not an eigenvalue ($\det\neq0$).</details>
6. $A$ is $2\times2$ with eigenvalues 2 and $-\tfrac12$. Give $\det A$, $\operatorname{tr}A$, and the eigenvalues of
   $A^{-1}$, $A^2$ and $A - 2I$. Is $A - 2I$ invertible?
   <details><summary>Answer</summary>$\det A = -1$, $\operatorname{tr}A = 1.5$; $A^{-1}$: $\tfrac12, -2$; $A^2$: $4, \tfrac14$;
   $A - 2I$: $0, -\tfrac52$ — not invertible, because 0 is now an eigenvalue.</details>
7. Prove that eigenvectors belonging to different eigenvalues of a **symmetric** matrix are orthogonal.
   <details><summary>Answer</summary>$\lambda_1\mathbf x_1^\top\mathbf x_2 = (A\mathbf x_1)^\top\mathbf x_2 = \mathbf x_1^\top A^\top\mathbf x_2 =
   \mathbf x_1^\top A\mathbf x_2 = \lambda_2\mathbf x_1^\top\mathbf x_2$, so $(\lambda_1-\lambda_2)\mathbf x_1^\top\mathbf x_2 = 0$; since
   $\lambda_1\ne\lambda_2$, $\mathbf x_1^\top\mathbf x_2 = 0$.</details>
8. Show that $X^\top X$ is positive semidefinite for any data matrix $X$, and explain why for $N = 24$ samples and
   $J = 400$ genes it has at least 376 zero eigenvalues.
   <details><summary>Answer</summary>$\mathbf w^\top X^\top X\mathbf w = \|X\mathbf w\|^2\ge0$. $\operatorname{rank}(X^\top X) =
   \operatorname{rank}X\le\min(24, 400) = 24$; a symmetric (diagonalizable) matrix has as many non-zero eigenvalues as its
   rank, so at least $400 - 24 = 376$ eigenvalues are 0.</details>
9. Power iteration on $\begin{pmatrix}1&2\\3&2\end{pmatrix}$ gains a factor 4 in accuracy per step. Why 4? Why did it fail
   on the adjacency matrix of the inflammation network, and why did adding $I$ fix it?
   <details><summary>Answer</summary>The error shrinks by $|\lambda_2/\lambda_1| = |-1/4|$ per step. The network's spectrum
   has $\lambda_{\max} = 2.37$ and $\lambda_{\min} = -2.37$ — equal absolute values, so neither component fades and the
   iterates oscillate. $A + I$ has the same eigenvectors and eigenvalues $3.37$ and $-1.37$: the tie is broken.</details>
10. For the insect Leslie matrix with both survival probabilities equal to $s$, find $s$ such that $\lambda_1 = 1$.
    <details><summary>Answer</summary>Expanding along column 3: $\det(L-\lambda I) = 8s^2 - \lambda(\lambda^2 - 6s) =
    -(\lambda^3 - 6s\lambda - 8s^2)$. At $\lambda = 1$: $1 - 6s - 8s^2 = 0$, $s = \frac{-6+\sqrt{36+32}}{16} = \frac{-6 + \sqrt{68}}{16}
    \approx 0.140$. With survival above 14 % per year the population grows.</details>
11. Rotation by 90°, $\begin{pmatrix}0&-1\\1&0\end{pmatrix}$: what are its eigenvalues, and what does the answer mean
    geometrically?
    <details><summary>Answer</summary>$f(\lambda) = \lambda^2 + 1$, so $\lambda = \pm i$. No real eigenvalue — the rotation
    turns **every** direction, so no line through the origin is mapped to itself. $|\lambda| = 1$: lengths are kept.</details>

## 🧠 Questions & discussion

1. The stable age distribution of a Leslie model is reached "whatever the starting counts". Which biological events
   (a harsh winter, harvesting of adults, an introduced predator) would repeatedly knock a real population away from
   it, and what would you then observe in yearly counts?
2. Eigenvector centrality ranked NFKB1 first and gave the interferon module zero. Is a highly central gene necessarily
   a good drug target? What other information would you want (expression, essentiality, direction of the links)?
3. The covariance eigenvector of STAT1 and MX1 mostly separates infected from control samples. When could the leading
   eigenvector of a gene–gene covariance matrix reflect something technical instead (batch, RNA quality)?
4. Eigenvalues of real matrices can be complex. In a biological dynamical system (e.g. a predator–prey or a
   cell-cycle oscillator), what would complex eigenvalues mean for the trajectories?
5. A $20{,}000\times20{,}000$ gene–gene covariance matrix from 50 samples has at most 49 non-zero eigenvalues. What does
   that imply for estimating gene–gene relationships from few samples?

## 🚀 Bonus — going further

- **Sensitivity of the growth rate.** For a Leslie matrix, $\partial\lambda_1/\partial L_{ij} = w_i v_j/(\mathbf w^\top\mathbf v)$,
  where $\mathbf v$ is the right and $\mathbf w$ the left dominant eigenvector (Caswell 2001). Compute it for the 4-class
  example and check it against finite differences: which vital rate should a conservation programme target?
- **Inverse iteration.** Power iteration on $(A - \mu I)^{-1}$ converges to the eigenvector whose eigenvalue is closest to
  $\mu$ (Section 5.4). Use it to find the eigenvector of $B$ for $\lambda = 1$ starting from $\mu = 0.9$.
- **Gershgorin discs.** Every eigenvalue lies in a disc centred at some $a_{ii}$ with radius $\sum_{j\ne i}|a_{ij}|$. Draw the
  discs for $B$ and the five maps and mark the eigenvalues.
- **Interactive textbook.** Margalit & Rabinoff, *Interactive Linear Algebra* (Georgia Tech,
  [textbooks.math.gatech.edu/ila](https://textbooks.math.gatech.edu/ila/)), chapter 5, lets you drag vectors and watch
  $A\mathbf x$ — a good companion to Sections 1 and 4.
- **Fibonacci rabbits.** Fibonacci's rabbit model is $\mathbf n_{t+1} = \begin{pmatrix}1&1\\1&0\end{pmatrix}\mathbf n_t$. Show
  that the dominant eigenvalue is the golden ratio $(1+\sqrt5)/2$ and explain why $F_{t+1}/F_t$ converges to it.

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module: 12 — Eigenvalues and eigenvectors
Input: small typed-in matrices (2×2, 3×3, 4×4), illustrative Leslie matrices, SIMULATED expression of STAT1/MX1
       (24 samples), the illustrative inflammation network (14 genes), DNA Markov chains (Durbin et al. 1998)
Output: characteristic polynomials, eigenvalues, eigenvectors/eigenspaces, growth rates, stable age structures,
        centrality scores, stationary base compositions
Method: det(A − λI) = 0 → roots → null space of A − λI; SymPy for exact work, eig/eigh numerically, power iteration
Key equation: A x = λ x  ⇔  (A − λI) x = 0 with A − λI singular  ⇔  det(A − λI) = 0
Key parameters: matrix entries (fecundity, survival); number of power-iteration steps; shift c in A + cI
What changed when I modified … (survival s in Section 9, D34 in Section 6, the gene pair in Section 7):
One failure mode: (e.g. power iteration oscillates when |λmax| = |λmin|; VΛVᵀ ≠ A with unnormalised eigenvectors)
Connection to 02-680: definition and verification, characteristic polynomial, 2×2 and 3×3 examples, the five maps,
       independence, det = Πλ, tr = Σλ, triangular matrices, symmetric matrices and the spectral theorem, BᵀB ⪰ 0
```

## 📚 References

- Deisenroth M. P., Faisal A. A., Ong C. S. (2020) *Mathematics for Machine Learning*. Cambridge University Press —
  §4.1–4.2 (determinant, trace, eigenvalues and eigenvectors; the overview of five linear maps), §4.4 (spectral theorem).
- Strang G. (2016) *Introduction to Linear Algebra*, 5th ed. Wellesley-Cambridge Press — Chapter 6 (eigenvalues and
  eigenvectors, symmetric and positive definite matrices).
- Isaak S., Manougian M. N. (1976) *Basic Concepts of Linear Algebra*. W. W. Norton — §7 (characteristic values and
  vectors).
- Betti M. *Mathematics and Statistics for the Quantitative Sciences* — the chapter on eigenvalues and linear dynamical
  systems.
- Liben-Nowell D. (2022) *Connecting Discrete Mathematics and Computer Science*, 2nd ed. Cambridge University Press —
  matrices and their applications.
- Kolter Z. *Linear Algebra Review and Reference* (CMU), §3.12–3.13 (eigenvalues, eigenvectors, symmetric matrices).
- Margalit D., Rabinoff J. *Interactive Linear Algebra*. Georgia Institute of Technology — Chapter 5.
- Leslie P. H. (1945) On the use of matrices in certain population mathematics. *Biometrika* 33:183–212.
- Caswell H. (2001) *Matrix Population Models: Construction, Analysis, and Interpretation*, 2nd ed. Sinauer.
- Durbin R., Eddy S. R., Krogh A., Mitchison G. (1998) *Biological Sequence Analysis*. Cambridge University Press —
  Chapter 3 (Markov chains and CpG islands; the transition tables the course's DNA simulator rounds).
- Brin S., Page L. (1998) The anatomy of a large-scale hypertextual Web search engine. *Computer Networks and ISDN
  Systems* 30:107–117.
- D. DeBlasio, 02-680 lecture notes, github.com/deblasiolab/02-680_Documents.

## ➡️ Next

[Module 13 · 🧩 Diagonalization and the singular value decomposition](13_diagonalization_svd.ipynb) turns this module's eigenvectors into a **factorisation**. When a matrix has $n$ independent eigenvectors it can
be written $A = X\Lambda X^{-1}$, which makes powers ($L^t$ for the population, $P^t$ for Markov chains) a matter of
raising numbers to powers. Then it generalises the idea to *rectangular* data matrices — 24 samples × 400 genes — with
the **singular value decomposition**, which finds the expression patterns that separate tissues and infection status
and underlies PCA.